# Libraries

In [80]:
import pandas as pd
import numpy as np
import shapefile
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from joblib import Parallel, delayed
from pathlib import Path
import geopandas as gpd
from sklearn.linear_model import LinearRegression
import seaborn as sns
from sklearn.linear_model import GammaRegressor
import contextily as cx
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.patches as mpatches
from sklearn.metrics import r2_score
import os
import gc
import pickle
from sklearn.linear_model import TweedieRegressor
from statsmodels.nonparametric.smoothers_lowess import lowess
import time
from decimal import Decimal, getcontext, InvalidOperation

# Upload Files

In [22]:
Proximity_164 = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Proximity_164_Final.csv")
Proximity_197 = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Proximity_197_Final.csv")
Proximity_328 = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Proximity_328_Final.csv")

In [64]:
Jamaica_Bay_Residential_Parcels = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Jamaica_Bay_Parcels_Set_Up_COMPLETE_Correction_DECADE_INCOMEREPLACEADJUST.csv")

In [ ]:
Part_One_245_Average_Results = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Part_One_245_Results_Premiums.csv")

In [24]:
all_columns = Jamaica_Bay_Residential_Parcels.columns

# Columns without either '245' or '585'
shared_columns = [col for col in all_columns if '245' not in col and '585' not in col]

# Columns specific to each scenario
columns_245 = [col for col in all_columns if '245' in col]
columns_585 = [col for col in all_columns if '585' in col]

# Create new dataframes
Part_Two_245_Shell = Jamaica_Bay_Residential_Parcels[shared_columns + columns_245].copy()
Part_Two_585_Shell = Jamaica_Bay_Residential_Parcels[shared_columns + columns_585].copy()

In [25]:
premium_columns = ['SBL', 'Year', 'NFIP_Premium', 'NFIP_Private_Premium', 'Alt_Premium', 'Alt_Private_Premium']

# Merge only selected columns
Part_Two_245_Shell = Part_Two_245_Shell.merge(
    Part_One_245_Average_Results[premium_columns],
    on=['SBL', 'Year'],
    how='left'
)

In [26]:
Part_Two_245_Shell.columns = [
    col.replace('_245', '') if '_245' in col else col
    for col in Part_Two_245_Shell.columns
]

Part_Two_585_Shell.columns = [
    col.replace('_585', '') if '_585' in col else col
    for col in Part_Two_585_Shell.columns
]

In [27]:
Part_Two_245_Shell = Part_Two_245_Shell.drop(columns='ReplaceCost')
Part_Two_245_Shell = Part_Two_245_Shell.rename(columns={
    'ReplaceCost_Adjusted': 'ReplaceCost'
})

Part_Two_585_Shell = Part_Two_585_Shell.drop(columns='ReplaceCost')
Part_Two_585_Shell = Part_Two_585_Shell.rename(columns={
    'ReplaceCost_Adjusted': 'ReplaceCost'
})

In [28]:
columns_to_drop = [
    'Unnamed: 0',
    'X',
    'YR_BLT',
    'BLDG_STYLE',
    'Income_Total',
    'Income_Total_Adjusted',
    'FIRM_FLD_ZONE',
    'PFIRM_FLD_ZONE',
    'BLD_STORY',
    'Land_MKT_VAL',
    'Total_MKT_VAL',
    'z_grade',
    'z_floor',
    'subgrade',
    'BFE',
    'DDF_Assign',
    'DDF_Dam',
    'Mu',
    'Sigma',
    'Income_House',
    'Std_Dev_Forecast',
    'County_FIPS',
    'County',
    'Income_Adjustment',
    'Zone_Decade'
]

Part_Two_245_Shell = Part_Two_245_Shell.drop(columns=columns_to_drop)

In [31]:
columns_to_drop = [
    'Unnamed: 0',
    'X',
    'YR_BLT',
    'BLDG_STYLE',
    'Income_Total',
    'Income_Total_Adjusted',
    'FIRM_FLD_ZONE',
    'PFIRM_FLD_ZONE',
    'BLD_STORY',
    'Land_MKT_VAL',
    'Total_MKT_VAL',
    'z_grade',
    'z_floor',
    'subgrade',
    'BFE',
    'DDF_Assign',
    'DDF_Dam',
    'Mu',
    'Sigma',
    'Income_House',
    'Std_Dev_Forecast',
    'County_FIPS',
    'County',
    'Income_Adjustment',
    'Zone_Decade'
]

Part_Two_585_Shell = Part_Two_585_Shell.drop(columns=columns_to_drop)

In [32]:
Part_Two_245_Shell.columns

Index(['SBL', 'Neighborhood', 'Census_Tract', 'Year', 'Mean_Forecast',
       'ReplaceCost', 'SLR', 'Prox_00005', 'Prox_00010', 'Prox_00030',
       'Prox_00050', 'Prox_00100', 'Prox_00300', 'Prox_00500', 'Prox_01000',
       'Prox_10000', 'SLR_Factor', 'Flood_Factor', 'Dam_Con_00005',
       'Dam_Con_00010', 'Dam_Con_00030', 'Dam_Con_00050', 'Dam_Con_00100',
       'Dam_Con_00300', 'Dam_Con_00500', 'Dam_Con_01000', 'Dam_Con_10000',
       'Dam_Struc_00005', 'Dam_Struc_00010', 'Dam_Struc_00030',
       'Dam_Struc_00050', 'Dam_Struc_00100', 'Dam_Struc_00300',
       'Dam_Struc_00500', 'Dam_Struc_01000', 'Dam_Struc_10000',
       'External_Factor', 'Initial_Market_Value', 'NFIP_Premium',
       'NFIP_Private_Premium', 'Alt_Premium', 'Alt_Private_Premium'],
      dtype='object')

# NFIP Set Up

## 245 Set Up

In [33]:
Part_Two_245_NFIP_Shell = Part_Two_245_Shell.copy()

new_cols = [
    'Flood_Event', 'Occupy_NFIP', 'Dam_T_Con', 'Dam_T_Struc', 'Dam_T', 'Dam_NFIP_Con','Dam_NFIP_Struc','Dam_NFIP',
    'Dam_NFIP_Deduct', 'Dam_NFIP_Private', 'Dam_NFIP_Private_Deduct', 'Flood_Occur_0_Factor', 'Flood_Occur_1_Factor',
    'Flood_Occur_2_Factor', 'Flood_Occur_3_Factor', 'Flood_Occur_4_Factor', 'Flood_Occur_Factor', 'Prox_Flood_0_Factor',
    'Prox_Flood_1_Factor', 'Prox_Flood_2_Factor', 'Prox_Flood_3_Factor', 'Prox_Flood_4_Factor', 'Prox_Flood_Factor',
    'Market_Value_Avg_NFIP', 'Cost_NFIP', 'Cost_NFIP_Deduct', 'Cost_NFIP_Private', 'Cost_NFIP_Private_Deduct', 
    'Dam_T_Con_00005', 'Dam_T_Con_00010', 'Dam_T_Con_00030', 'Dam_T_Con_00050', 'Dam_T_Con_00100',
    'Dam_T_Con_00300', 'Dam_T_Con_00500', 'Dam_T_Con_01000', 'Dam_T_Con_10000',
    'Dam_T_Struc_00005', 'Dam_T_Struc_00010', 'Dam_T_Struc_00030', 'Dam_T_Struc_00050',
    'Dam_T_Struc_00100', 'Dam_T_Struc_00300', 'Dam_T_Struc_00500', 'Dam_T_Struc_01000', 'Dam_T_Struc_10000',
    'Dam_T_00005', 'Dam_T_00010', 'Dam_T_00030', 'Dam_T_00050', 'Dam_T_00100',
    'Dam_T_00300', 'Dam_T_00500', 'Dam_T_01000', 'Dam_T_10000',
    'Dam_NFIP_Con_00005', 'Dam_NFIP_Con_00010', 'Dam_NFIP_Con_00030', 'Dam_NFIP_Con_00050', 'Dam_NFIP_Con_00100',
    'Dam_NFIP_Con_00300', 'Dam_NFIP_Con_00500', 'Dam_NFIP_Con_01000', 'Dam_NFIP_Con_10000',
    'Dam_NFIP_Struc_00005', 'Dam_NFIP_Struc_00010', 'Dam_NFIP_Struc_00030', 'Dam_NFIP_Struc_00050',
    'Dam_NFIP_Struc_00100', 'Dam_NFIP_Struc_00300', 'Dam_NFIP_Struc_00500', 'Dam_NFIP_Struc_01000',
    'Dam_NFIP_Struc_10000',
    'Dam_NFIP_00005', 'Dam_NFIP_00010', 'Dam_NFIP_00030', 'Dam_NFIP_00050', 'Dam_NFIP_00100',
    'Dam_NFIP_00300', 'Dam_NFIP_00500', 'Dam_NFIP_01000', 'Dam_NFIP_10000',
    'Dam_NFIP_Deduct_00005', 'Dam_NFIP_Deduct_00010', 'Dam_NFIP_Deduct_00030', 'Dam_NFIP_Deduct_00050',
    'Dam_NFIP_Deduct_00100', 'Dam_NFIP_Deduct_00300', 'Dam_NFIP_Deduct_00500', 'Dam_NFIP_Deduct_01000',
    'Dam_NFIP_Deduct_10000',
    'Dam_NFIP_Private_00005', 'Dam_NFIP_Private_00010', 'Dam_NFIP_Private_00030', 'Dam_NFIP_Private_00050',
    'Dam_NFIP_Private_00100', 'Dam_NFIP_Private_00300', 'Dam_NFIP_Private_00500', 'Dam_NFIP_Private_01000',
    'Dam_NFIP_Private_10000',
    'Dam_NFIP_Private_Deduct_00005', 'Dam_NFIP_Private_Deduct_00010', 'Dam_NFIP_Private_Deduct_00030',
    'Dam_NFIP_Private_Deduct_00050', 'Dam_NFIP_Private_Deduct_00100', 'Dam_NFIP_Private_Deduct_00300',
    'Dam_NFIP_Private_Deduct_00500', 'Dam_NFIP_Private_Deduct_01000', 'Dam_NFIP_Private_Deduct_10000',
    'Flood_Occur_0_00005_Factor', 'Flood_Occur_0_00010_Factor', 'Flood_Occur_0_00030_Factor',
    'Flood_Occur_0_00050_Factor', 'Flood_Occur_0_00100_Factor', 'Flood_Occur_0_00300_Factor',
    'Flood_Occur_0_00500_Factor', 'Flood_Occur_0_01000_Factor', 'Flood_Occur_0_10000_Factor',
    'Flood_Occur_00000_Factor', 'Flood_Occur_00005_Factor', 'Flood_Occur_00010_Factor',
    'Flood_Occur_00030_Factor', 'Flood_Occur_00050_Factor', 'Flood_Occur_00100_Factor',
    'Flood_Occur_00300_Factor', 'Flood_Occur_00500_Factor', 'Flood_Occur_01000_Factor',
    'Flood_Occur_10000_Factor',
    'Prox_Flood_0_00005_Factor', 'Prox_Flood_0_00010_Factor', 'Prox_Flood_0_00030_Factor',
    'Prox_Flood_0_00050_Factor', 'Prox_Flood_0_00100_Factor', 'Prox_Flood_0_00300_Factor',
    'Prox_Flood_0_00500_Factor', 'Prox_Flood_0_01000_Factor', 'Prox_Flood_0_10000_Factor',
    'Prox_Flood_00000_Factor', 'Prox_Flood_00005_Factor', 'Prox_Flood_00010_Factor',
    'Prox_Flood_00030_Factor', 'Prox_Flood_00050_Factor', 'Prox_Flood_00100_Factor',
    'Prox_Flood_00300_Factor', 'Prox_Flood_00500_Factor', 'Prox_Flood_01000_Factor',
    'Prox_Flood_10000_Factor',
    'Flood_Factor_00000', 'Flood_Factor_00005', 'Flood_Factor_00010', 'Flood_Factor_00030',
    'Flood_Factor_00050', 'Flood_Factor_00100', 'Flood_Factor_00300', 'Flood_Factor_00500',
    'Flood_Factor_01000', 'Flood_Factor_10000',
    'Market_Project_00000', 'Market_Project_00005', 'Market_Project_00010', 'Market_Project_00030', 'Market_Project_00050',
    'Market_Project_00100', 'Market_Project_00300', 'Market_Project_00500', 'Market_Project_01000', 'Market_Project_10000', 
    'DEU_NFIP', 'DEU_None', 'Decision_NFIP', 'Decision_None', 'NFIP_Premium_Paid', 'NFIP_Private_Premium_Paid',
    'Cost_None', 'Cost_NFIP_Premium', 'Cost_NFIP_Private_Premium', 'Dam_None'
]

Part_Two_245_NFIP_Shell = pd.concat(
    [Part_Two_245_NFIP_Shell,
     pd.DataFrame(np.nan, index=Part_Two_245_NFIP_Shell.index, columns=new_cols)],
    axis=1
)

In [34]:
# Remove 2024
Part_Two_245_NFIP_Shell = Part_Two_245_NFIP_Shell[Part_Two_245_NFIP_Shell["Year"] != 2024]

In [36]:
Part_Two_245_NFIP_Shell['Dam_T_Con_00005'] = (Part_Two_245_NFIP_Shell['Dam_Con_00005'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00010'] = (Part_Two_245_NFIP_Shell['Dam_Con_00010'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00030'] = (Part_Two_245_NFIP_Shell['Dam_Con_00030'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00050'] = (Part_Two_245_NFIP_Shell['Dam_Con_00050'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00100'] = (Part_Two_245_NFIP_Shell['Dam_Con_00100'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00300'] = (Part_Two_245_NFIP_Shell['Dam_Con_00300'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_00500'] = (Part_Two_245_NFIP_Shell['Dam_Con_00500'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)    
    
Part_Two_245_NFIP_Shell['Dam_T_Con_01000'] = (Part_Two_245_NFIP_Shell['Dam_Con_01000'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Con_10000'] = (Part_Two_245_NFIP_Shell['Dam_Con_10000'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'] * 0.5)
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00005'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00005'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00010'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00010'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00030'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00030'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00050'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00050'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00100'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00100'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00300'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00300'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_00500'] = (Part_Two_245_NFIP_Shell['Dam_Struc_00500'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])    
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_01000'] = (Part_Two_245_NFIP_Shell['Dam_Struc_01000'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])
    
Part_Two_245_NFIP_Shell['Dam_T_Struc_10000'] = (Part_Two_245_NFIP_Shell['Dam_Struc_10000'] *
    Part_Two_245_NFIP_Shell['ReplaceCost'])

Part_Two_245_NFIP_Shell['Dam_T_00005'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00005'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00005'])
    
Part_Two_245_NFIP_Shell['Dam_T_00010'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00010'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00010'])
    
Part_Two_245_NFIP_Shell['Dam_T_00030'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00030'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00030'])
    
Part_Two_245_NFIP_Shell['Dam_T_00050'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00050'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00050'])
    
Part_Two_245_NFIP_Shell['Dam_T_00100'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00100'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00100'])
    
Part_Two_245_NFIP_Shell['Dam_T_00300'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00300'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00300'])
    
Part_Two_245_NFIP_Shell['Dam_T_00500'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_00500'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_00500'])    
    
Part_Two_245_NFIP_Shell['Dam_T_01000'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_01000'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_01000'])
    
Part_Two_245_NFIP_Shell['Dam_T_10000'] = (Part_Two_245_NFIP_Shell['Dam_T_Con_10000'] +
    Part_Two_245_NFIP_Shell['Dam_T_Struc_10000'])
    
conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00005'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00005'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00005']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00005'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00010'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00010'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00010']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00010'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00030'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00030'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00030']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00030'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00050'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00050'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00050']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00050'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00100'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00100'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00100']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00100'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00300'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00300'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00300']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00300'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00500'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_00500'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_00500']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00500'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_01000'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_01000'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_01000']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_01000'] = np.select(conditions, choices, default=default)

conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Con_10000'] >= 111111.11),
        (Part_Two_245_NFIP_Shell['Dam_T_Con_10000'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Con_10000']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Con_10000'] = np.select(conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00005'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00005'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00005']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00005'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00010'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00010'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00010']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00010'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00030'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00030'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00030']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00030'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00050'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00050'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00050']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00050'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00100'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00100'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00100']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00100'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00300'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00300'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00300']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00300'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00500'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_00500'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_00500']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00500'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_01000'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_01000'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_01000']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_01000'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_10000'] >= 277777.78),
        (Part_Two_245_NFIP_Shell['Dam_T_Struc_10000'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Part_Two_245_NFIP_Shell['Dam_T_Struc_10000']
]

default = 0

Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_10000'] = np.select(Conditions, choices, default=default)

Part_Two_245_NFIP_Shell['Dam_NFIP_00005'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00005'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00005'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00010'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00010'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00010'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00030'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00030'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00030'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00050'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00050'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00050'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00100'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00100'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00100'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00300'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00300'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00300'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00500'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00500'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00500'])

Part_Two_245_NFIP_Shell['Dam_NFIP_00100'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_00100'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_00100'])

Part_Two_245_NFIP_Shell['Dam_NFIP_01000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_01000'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_01000'])

Part_Two_245_NFIP_Shell['Dam_NFIP_10000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Con_10000'] + Part_Two_245_NFIP_Shell['Dam_NFIP_Struc_10000'])

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00005'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00005'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00010'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00010'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00030'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00030'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00050'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00050'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00100'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00100'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00300'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00300'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00500'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_00500'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_01000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_01000'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_10000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_10000'] * (1/9))
    
Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00005'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00005'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00005'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00005'])) 

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00010'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00010'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00010'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00010']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00030'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00030'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00030'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00030']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00050'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00050'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00050'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00050']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00100'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00100'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00100'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00100']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00300'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00300'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00300'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00300']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00500'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_00500'] - Part_Two_245_NFIP_Shell['Dam_NFIP_00500'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_00500']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_01000'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_01000'] - Part_Two_245_NFIP_Shell['Dam_NFIP_01000'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_01000']))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_10000'] = (0.9 * (Part_Two_245_NFIP_Shell['Dam_T_10000'] - Part_Two_245_NFIP_Shell['Dam_NFIP_10000'] - Part_Two_245_NFIP_Shell['Dam_NFIP_Deduct_10000']))
    
Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00005'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00005'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00010'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00010'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00030'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00030'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00050'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00050'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00100'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00100'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00300'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00300'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_00500'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_00500'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_01000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_01000'] * (1/9))

Part_Two_245_NFIP_Shell['Dam_NFIP_Private_Deduct_10000'] = (Part_Two_245_NFIP_Shell['Dam_NFIP_Private_10000'] * (1/9))
    
Part_Two_245_NFIP_Shell['Flood_Occur_0_00005_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00005'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00010_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00010'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00030_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00030'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00050_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00050'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00100_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00100'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00300_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00300'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_00500_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_00500'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_01000_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_01000'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Flood_Occur_0_10000_Factor'] = np.where(
        Part_Two_245_NFIP_Shell['Dam_T_10000'] > 0,
        0.805,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00005_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00005'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00010_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00010'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00030_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00030'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00050_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00050'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00100_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00100'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00300_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00300'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_00500_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_00500'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_01000_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_01000'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Prox_Flood_0_10000_Factor'] = np.where(
        (Part_Two_245_NFIP_Shell['Prox_10000'] == 'Y'),
        0.935,
        1
    )

Part_Two_245_NFIP_Shell['Occupy_NFIP'] = np.where(Part_Two_245_NFIP_Shell['SLR'] < Part_Two_245_NFIP_Shell['Year'], 0, 1)


Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Flood_Occur_1_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Flood_Occur_2_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Flood_Occur_3_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Flood_Occur_4_Factor"] = 1

mask_2 = (
    (Part_Two_245_NFIP_Shell["Year"] == 2025) &
    (
        (Part_Two_245_NFIP_Shell["Dam_Con_00005"] > 0) |
        (Part_Two_245_NFIP_Shell["Dam_Struc_00005"] > 0)
    )
)
Part_Two_245_NFIP_Shell.loc[mask_2, "Flood_Occur_2_Factor"] = 0.883

mask_3 = mask_2.copy()  # same condition
Part_Two_245_NFIP_Shell.loc[mask_3, "Flood_Occur_3_Factor"] = 0.922

mask_4 = (
    (Part_Two_245_NFIP_Shell["Year"] == 2025) &
    (
        (Part_Two_245_NFIP_Shell["Dam_Con_00010"] > 0) |
        (Part_Two_245_NFIP_Shell["Dam_Struc_00010"] > 0)
    )
)
Part_Two_245_NFIP_Shell.loc[mask_4, "Flood_Occur_4_Factor"] = 0.961

Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Prox_Flood_1_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Prox_Flood_2_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Prox_Flood_3_Factor"] = 1
Part_Two_245_NFIP_Shell.loc[Part_Two_245_NFIP_Shell["Year"] == 2025, "Prox_Flood_4_Factor"] = 1

# Override only where condition is met
mask_2 = (Part_Two_245_NFIP_Shell["Year"] == 2025) & (Part_Two_245_NFIP_Shell["Prox_00005"] == "Y")
Part_Two_245_NFIP_Shell.loc[mask_2, "Prox_Flood_2_Factor"] = 0.961

mask_3 = (Part_Two_245_NFIP_Shell["Year"] == 2025) & (Part_Two_245_NFIP_Shell["Prox_00005"] == "Y")
Part_Two_245_NFIP_Shell.loc[mask_3, "Prox_Flood_3_Factor"] = 0.974

mask_4 = (Part_Two_245_NFIP_Shell["Year"] == 2025) & (Part_Two_245_NFIP_Shell["Prox_00010"] == "Y")
Part_Two_245_NFIP_Shell.loc[mask_4, "Prox_Flood_4_Factor"] = 0.987


In [ ]:
Part_Two_245_NFIP_Shell.to_csv('Part_Two_245_NFIP_Shell.csv', index=False)

In [139]:
tmp = Part_Two_245_NFIP_Shell[(Part_Two_245_NFIP_Shell['SBL'].astype(str).isin(["4142431227", "3011340029", "4142431110", "3036180001", "3036250007", "3086160111", "3080580018", "3086320028"]))]

In [89]:
tmp = Part_Two_245_NFIP_Shell.sort_values(['SBL', 'Year'])

In [142]:
tmp = tmp.sort_values(['SBL', 'Year'])
df = tmp.copy()

getcontext().prec = 15

intensity = ['00000', '00005', '00010', '00030', '00050', '00100', '00300', '00500', '01000', '10000']
weights = {k: 1 / int(k) for k in intensity if k != '00000'}
weights['00000'] = 1 - sum(weights.values())

start_time = time.time()

for year in range(2025, 2076):  # Year = 2025 to 2075 inclusive
    year_start = time.time()

    random_number = np.random.randint(0, 10001)
    if random_number < 6303:
        df.loc[df['Year'] == year, 'Flood_Event'] = 0
    elif 6303 <= random_number < 8303:
        df.loc[df['Year'] == year, 'Flood_Event'] = 5
    elif 8303 <= random_number < 9303:
        df.loc[df['Year'] == year, 'Flood_Event'] = 10
    elif 9303 <= random_number < 9636:
        df.loc[df['Year'] == year, 'Flood_Event'] = 30
    elif 9636 <= random_number < 9836:
        df.loc[df['Year'] == year, 'Flood_Event'] = 50
    elif 9836 <= random_number < 9936:
        df.loc[df['Year'] == year, 'Flood_Event'] = 100
    elif 9936 <= random_number < 9969:
        df.loc[df['Year'] == year, 'Flood_Event'] = 300
    elif 9969 <= random_number < 9989:
        df.loc[df['Year'] == year, 'Flood_Event'] = 500
    elif 9989 <= random_number < 9999:
        df.loc[df['Year'] == year, 'Flood_Event'] = 1000
    else:
        df.loc[df['Year'] == year, 'Flood_Event'] = 10000

    for sbl in df.loc[df['Year'] == year, 'SBL'].unique():
        sbl_start = time.time()
        print(f"\nProcessing SBL: {sbl}")
        
        sbl_df = df[df['SBL'] == sbl]
        base_row = sbl_df[sbl_df['Year'] == year]

        if base_row.empty:
            continue
            
        occupy = base_row['Occupy_NFIP'].values[0]

        if occupy == 0:
            # Still need to update cost columns with zero values
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), [
                'Dam_T', 'Dam_T_Con', 'Dam_T_Struc',
                'Dam_NFIP', 'Dam_NFIP_Con', 'Dam_NFIP_Struc', 'Dam_NFIP_Deduct',
                'Dam_NFIP_Private', 'Dam_NFIP_Private_Deduct', 'Dam_None',
                'NFIP_Premium_Paid', 'NFIP_Private_Premium_Paid',
                'Decision_NFIP', 'Decision_None'
            ]] = 0
            continue
            
        flood_occur_base_factors = [
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), f'Flood_Occur_{i}_Factor'].fillna(1).values[0]
            for i in range(1, 5)
        ]
        flood_occur_min = min(flood_occur_base_factors)
        
        prox_flood_base_factors = [
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), f'Prox_Flood_{i}_Factor'].fillna(1).values[0]
            for i in range(1, 5)
        ]
        prox_flood_min = min(prox_flood_base_factors)

        deu_none = 0
        deu_nfip = 0

        slr_year = df.loc[df['SBL'] == sbl, 'SLR'].dropna().values[0]
        end_year = min(year + 30, slr_year + 1)  # Limit the range to SLR + 1 at most

        for suffix in intensity:
            npv_values_none = []
            npv_values_nfip = []

            if suffix == '00000':
                for y in range(year, end_year):
                    df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Flood_Occur_00000_Factor'] = flood_occur_min
                    df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Prox_Flood_00000_Factor'] = prox_flood_min
                    df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Flood_Factor_00000'] = min(flood_occur_min, prox_flood_min)
            else:
                col_name = f'Flood_Occur_0_{suffix}_Factor'
                if col_name in df.columns:
                    for y in range(year, end_year):
                        val = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), col_name].fillna(1).values
                        if len(val) > 0:
                            flood_occur_val = min(flood_occur_min, val[0])
                            prox_col = f'Prox_Flood_0_{suffix}_Factor'
                            prox_val = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), prox_col].fillna(1).values

                            if len(prox_val) > 0:
                                prox_flood_val = min(prox_flood_min, prox_val[0])
                                df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Prox_Flood_{suffix}_Factor'] = prox_flood_val

                            df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Flood_Occur_{suffix}_Factor'] = flood_occur_val
                            df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Prox_Flood_{suffix}_Factor'] = prox_flood_val
                            df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Flood_Factor_{suffix}'] = min(flood_occur_val, prox_flood_val)

            for y in range(year, end_year):
                flood_factor = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Flood_Factor_{suffix}'].fillna(1).values[0]
                initial_val = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), 'Initial_Market_Value'].fillna(1).values[0]
                external_val = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), 'External_Factor'].fillna(1).values[0]
                slr_val = df.loc[(df['SBL'] == sbl) & (df['Year'] == y), 'SLR_Factor'].fillna(1).values[0]
                df.loc[(df['SBL'] == sbl) & (df['Year'] == y), f'Market_Project_{suffix}'] = (
                    initial_val * external_val * slr_val * flood_factor
                )

            window_df = df[(df['SBL'] == sbl) & (df['Year'] >= year) & (df['Year'] < end_year)]

            for t, (_, row) in enumerate(window_df.iterrows(), start=1):
                market_val = Decimal(str(row.get(f'Market_Project_{suffix}', 0)))
                dam_val_none = Decimal(str(row.get(f'Dam_T_{suffix}', 0)))
                dam_nfip_deduct = Decimal(str(row.get(f'Dam_NFIP_Deduct_{suffix}', 0)))
                dam_nfip_private_deduct = Decimal(str(row.get(f'Dam_NFIP_Private_Deduct_{suffix}', 0)))
                premium_nfip = Decimal(str(row.get('NFIP_Premium', 0)))
                premium_private = Decimal(str(row.get('NFIP_Private_Premium', 0)))
                replace_cost = Decimal(str(row.get('ReplaceCost', 0)))

                if replace_cost == 0:
                    npv_none = 0
                    npv_nfip = 0
                else:
                    exp_none = (-1 / (Decimal('0.05') * replace_cost)) * (market_val - dam_val_none)
                    exp_nfip = (-1 / (Decimal('0.05') * replace_cost)) * (
                        market_val - dam_nfip_deduct - dam_nfip_private_deduct - premium_nfip - premium_private
                    )
        
                    npv_none = (1 / (Decimal('1.03') ** t)) * (1 - exp_none.exp())
                    npv_nfip = (1 / (Decimal('1.03') ** t)) * (1 - exp_nfip.exp())

                npv_values_none.append(npv_none)
                npv_values_nfip.append(npv_nfip)

            weight = Decimal(str(weights.get(suffix, 0)))
            deu_none += weight * sum(npv_values_none)
            deu_nfip += weight * sum(npv_values_nfip)

        df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'DEU_None'] = deu_none
        df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'DEU_NFIP'] = deu_nfip
        
        print(f"SBL {sbl}, Year {year}: DEU_None = {deu_none}, DEU_NFIP = {deu_nfip}")

        nfip_ = df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'NFIP_Premium'].fillna(0).values[0]
        nfip_private = df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'NFIP_Private_Premium'].fillna(0).values[0]

        if deu_none >= deu_nfip:  
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'Decision_None'] = 1
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'Decision_NFIP'] = 0
        else:
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'Decision_None'] = 0
            df.loc[(df['SBL'] == sbl) & (df['Year'] == year), 'Decision_NFIP'] = 1

        zero_event_mask = (df['SBL'] == sbl) & (df['Year'] == year) & (df['Flood_Event'] == 0)

        df.loc[zero_event_mask, 'Dam_T_Con'] = 0
        df.loc[zero_event_mask, 'Dam_T_Struc'] = 0
        df.loc[zero_event_mask, 'Dam_T'] = 0
        df.loc[zero_event_mask, 'Flood_Occur_0_Factor'] = 1
        df.loc[zero_event_mask, 'Flood_Occur_Factor'] = df.loc[zero_event_mask, 'Flood_Occur_00000_Factor']
        df.loc[zero_event_mask, 'Prox_Flood_0_Factor'] = 1
        df.loc[zero_event_mask, 'Prox_Flood_Factor'] = df.loc[zero_event_mask, 'Prox_Flood_00000_Factor']
        df.loc[zero_event_mask, 'Flood_Factor'] = df.loc[zero_event_mask, 'Flood_Factor_00000']
        df.loc[zero_event_mask, 'Market_Value_Avg_NFIP'] = df.loc[zero_event_mask, 'Market_Project_00000']
        df.loc[zero_event_mask, 'Dam_NFIP_Con'] = 0
        df.loc[zero_event_mask, 'Dam_NFIP_Struc'] = 0
        df.loc[zero_event_mask, 'Dam_NFIP'] = 0
        df.loc[zero_event_mask, 'Dam_NFIP_Deduct'] = 0
        df.loc[zero_event_mask, 'Dam_NFIP_Private'] = 0
        df.loc[zero_event_mask, 'Dam_NFIP_Private_Deduct'] = 0
        df.loc[zero_event_mask, 'Dam_None'] = 0

        
        df.loc[zero_event_mask & (df['Decision_NFIP'] == 1), 'NFIP_Premium_Paid'] = df.loc[zero_event_mask & (df['Decision_NFIP'] == 1), 'NFIP_Premium']
        df.loc[zero_event_mask & (df['Decision_NFIP'] == 1), 'NFIP_Private_Premium_Paid'] = df.loc[zero_event_mask & (df['Decision_NFIP'] == 1), 'NFIP_Private_Premium']
        df.loc[zero_event_mask & (df['Decision_None'] == 1), 'NFIP_Premium_Paid'] = 0
        df.loc[zero_event_mask & (df['Decision_None'] == 1), 'NFIP_Private_Premium_Paid'] = 0

        
        mask = (df['SBL'] == sbl) & (df['Year'] == year)
        if df.loc[mask, 'SLR'].values[0] == year:
            replace_cost = df.loc[mask, 'ReplaceCost'].values[0]

            df.loc[mask, 'Dam_T_Con'] = 0.5 * replace_cost
            df.loc[mask, 'Dam_T_Struc'] = replace_cost
            df.loc[mask, 'Dam_T'] = 1.5 * replace_cost
            df.loc[flood_event_mask, 'Market_Value_Avg_NFIP'] = 0

            nfip_mask = mask & (df['Decision_NFIP'] == 1)
            if nfip_mask.any():
                dam_t_con = df.loc[nfip_mask, 'Dam_T_Con'].values[0]
                dam_t_struc = df.loc[nfip_mask, 'Dam_T_Struc'].values[0]
                dam_t = dam_t_con + dam_t_struc

                dam_nfip_con = 100000 if dam_t_con >= 111111.11 else 0.9 * dam_t_con
                dam_nfip_struc = 250000 if dam_t_struc >= 277777.78 else 0.9 * dam_t_struc
                dam_nfip = dam_nfip_con + dam_nfip_struc
                dam_nfip_deduct = dam_nfip * (1 / 9)
                dam_nfip_private = 0.9 * (dam_t - dam_nfip - dam_nfip_deduct)
                dam_nfip_private_deduct = (1 / 9) * dam_nfip_private

                df.loc[nfip_mask, 'Dam_NFIP_Con'] = dam_nfip_con
                df.loc[nfip_mask, 'Dam_NFIP_Struc'] = dam_nfip_struc
                df.loc[nfip_mask, 'Dam_NFIP'] = dam_nfip
                df.loc[nfip_mask, 'Dam_NFIP_Deduct'] = dam_nfip_deduct
                df.loc[nfip_mask, 'Dam_NFIP_Private'] = dam_nfip_private
                df.loc[nfip_mask, 'Dam_NFIP_Private_Deduct'] = dam_nfip_private_deduct
                df.loc[nfip_mask, 'Dam_None'] = 0
                df.loc[nfip_mask, 'NFIP_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Premium']
                df.loc[nfip_mask, 'NFIP_Private_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Private_Premium']

            none_mask = mask & (df['Decision_None'] == 1)
            if none_mask.any():
                df.loc[none_mask, 'Dam_NFIP_Con'] = 0
                df.loc[none_mask, 'Dam_NFIP_Struc'] = 0
                df.loc[none_mask, 'Dam_NFIP'] = 0
                df.loc[none_mask, 'Dam_NFIP_Deduct'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private_Deduct'] = 0
                df.loc[none_mask, 'Dam_None'] = df.loc[none_mask, 'Dam_T']
                df.loc[none_mask, 'NFIP_Premium_Paid'] = 0
                df.loc[none_mask, 'NFIP_Private_Premium_Paid'] = 0

        
        else:
            for suffix in intensity:
                if suffix == '00000':
                    continue
                event_val = int(suffix)
                flood_event_mask = mask & (df['Flood_Event'] == event_val)

                df.loc[flood_event_mask, 'Dam_T_Con'] = df.loc[flood_event_mask, f'Dam_T_Con_{suffix}']
                df.loc[flood_event_mask, 'Dam_T_Struc'] = df.loc[flood_event_mask, f'Dam_T_Struc_{suffix}']
                df.loc[flood_event_mask, 'Dam_T'] = df.loc[flood_event_mask, f'Dam_T_{suffix}']
                df.loc[flood_event_mask, 'Flood_Occur_0_Factor'] = df.loc[flood_event_mask, f'Flood_Occur_0_{suffix}_Factor']
                df.loc[flood_event_mask, 'Flood_Occur_Factor'] = df.loc[flood_event_mask, f'Flood_Occur_{suffix}_Factor']
                df.loc[flood_event_mask, 'Prox_Flood_0_Factor'] = df.loc[flood_event_mask, f'Prox_Flood_0_{suffix}_Factor']
                df.loc[flood_event_mask, 'Prox_Flood_Factor'] = df.loc[flood_event_mask, f'Prox_Flood_{suffix}_Factor']
                df.loc[flood_event_mask, 'Flood_Factor'] = df.loc[flood_event_mask, f'Flood_Factor_{suffix}']
                df.loc[flood_event_mask, 'Market_Value_Avg_NFIP'] = df.loc[flood_event_mask, f'Market_Project_{suffix}']

                nfip_mask = flood_event_mask & (df['Decision_NFIP'] == 1)
                df.loc[nfip_mask, 'Dam_NFIP_Con'] = df.loc[nfip_mask, f'Dam_NFIP_Con_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Struc'] = df.loc[nfip_mask, f'Dam_NFIP_Struc_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP'] = df.loc[nfip_mask, f'Dam_NFIP_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Deduct'] = df.loc[nfip_mask, f'Dam_NFIP_Deduct_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Private'] = df.loc[nfip_mask, f'Dam_NFIP_Private_{suffix}']
                df.loc[nfip_mask, 'Dam_NFIP_Private_Deduct'] = df.loc[nfip_mask, f'Dam_NFIP_Private_Deduct_{suffix}']
                df.loc[nfip_mask, 'Dam_None'] = 0
                df.loc[nfip_mask, 'NFIP_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Premium']
                df.loc[nfip_mask, 'NFIP_Private_Premium_Paid'] = df.loc[nfip_mask, 'NFIP_Private_Premium']

                none_mask = flood_event_mask & (df['Decision_None'] == 1)
                df.loc[none_mask, 'Dam_NFIP_Con'] = 0
                df.loc[none_mask, 'Dam_NFIP_Struc'] = 0
                df.loc[none_mask, 'Dam_NFIP'] = 0
                df.loc[none_mask, 'Dam_NFIP_Deduct'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private'] = 0
                df.loc[none_mask, 'Dam_NFIP_Private_Deduct'] = 0
                df.loc[none_mask, 'Dam_None'] = df.loc[none_mask, 'Dam_T']
                df.loc[none_mask, 'NFIP_Premium_Paid'] = 0
                df.loc[none_mask, 'NFIP_Private_Premium_Paid'] = 0
            
        df['Cost_NFIP'] = df.groupby('SBL')['Dam_NFIP'].cumsum()
        df['Cost_NFIP_Deduct'] = df.groupby('SBL')['Dam_NFIP_Deduct'].cumsum()
        df['Cost_NFIP_Private'] = df.groupby('SBL')['Dam_NFIP_Private'].cumsum()
        df['Cost_NFIP_Private_Deduct'] = df.groupby('SBL')['Dam_NFIP_Private_Deduct'].cumsum()
        df['Cost_None'] = df.groupby('SBL')['Dam_None'].cumsum()
        df['Cost_NFIP_Premium'] = df.groupby('SBL')['NFIP_Premium_Paid'].cumsum()
        df['Cost_NFIP_Private_Premium'] = df.groupby('SBL')['NFIP_Private_Premium_Paid'].cumsum()
                
        # Map current factors to next year's updates
        next_year = year + 1
        mask_next = (df['SBL'] == sbl) & (df['Year'] == next_year)
        mask_current = (df['SBL'] == sbl) & (df['Year'] == year)

        # Flood_Occur_1_Factor depends on Flood_Occur_0_Factor from current year
        flood_0 = df.loc[mask_current, 'Flood_Occur_0_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_1_Factor'] = np.where(flood_0 < 1, 0.844, 1)

        # Flood_Occur_2_Factor depends on Flood_Occur_1_Factor from current year
        flood_1 = df.loc[mask_current, 'Flood_Occur_1_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_2_Factor'] = np.where(flood_1 < 1, 0.883, 1)

        # Flood_Occur_3_Factor depends on Flood_Occur_2_Factor from current year
        flood_2 = df.loc[mask_current, 'Flood_Occur_2_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_3_Factor'] = np.where(flood_2 < 1, 0.922, 1)

        # Flood_Occur_4_Factor depends on Flood_Occur_3_Factor from current year
        flood_3 = df.loc[mask_current, 'Flood_Occur_3_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Flood_Occur_4_Factor'] = np.where(flood_3 < 1, 0.961, 1)

        # Prox_Flood_1_Factor depends on Prox_Flood_0_Factor from current year
        prox_0 = df.loc[mask_current, 'Prox_Flood_0_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_1_Factor'] = np.where(prox_0 < 1, 0.948, 1)

        # Prox_Flood_2_Factor depends on Prox_Flood_1_Factor from current year
        prox_1 = df.loc[mask_current, 'Prox_Flood_1_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_2_Factor'] = np.where(prox_1 < 1, 0.961, 1)

        # Prox_Flood_3_Factor depends on Prox_Flood_2_Factor from current year
        prox_2 = df.loc[mask_current, 'Prox_Flood_2_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_3_Factor'] = np.where(prox_2 < 1, 0.974, 1)

        # Prox_Flood_4_Factor depends on Prox_Flood_3_Factor from current year
        prox_3 = df.loc[mask_current, 'Prox_Flood_3_Factor'].fillna(1).values[0]
        df.loc[mask_next, 'Prox_Flood_4_Factor'] = np.where(prox_3 < 1, 0.987, 1) 

        sbl_duration = time.time() - sbl_start
        print(f"Finished SBL: {sbl} in {sbl_duration:.2f} seconds")

    year_duration = time.time() - year_start
    print(f"Year {year} processed in {year_duration:.2f} seconds")

total_duration = time.time() - start_time
print(f"\nTotal time for all SBLs and years: {total_duration:.2f} seconds")


Processing SBL: 3011340029
SBL 3011340029, Year 2025: DEU_None = 19.6004413492429, DEU_NFIP = 19.6004413492427
Finished SBL: 3011340029 in 3.36 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2025: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.43 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2025: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036250007 in 3.44 seconds

Processing SBL: 3080580018
SBL 3080580018, Year 2025: DEU_None = 19.6004413484797, DEU_NFIP = 19.6004413493829
Finished SBL: 3080580018 in 3.17 seconds

Processing SBL: 3086160111
SBL 3086160111, Year 2025: DEU_None = 19.6004413492480, DEU_NFIP = 19.6004413492479
Finished SBL: 3086160111 in 3.13 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2025: DEU_None = 19.6004189104402, DEU_NFIP = 19.6004413445551
Finished SBL: 3086320028 in 3.31 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2025: DEU_None = 19.5975512392549

SBL 3086160111, Year 2031: DEU_None = 19.6004413490989, DEU_NFIP = 19.6004413490985
Finished SBL: 3086160111 in 3.43 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2031: DEU_None = 19.6004035482269, DEU_NFIP = 19.6004413410675
Finished SBL: 3086320028 in 3.57 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2031: DEU_None = 19.5932506535455, DEU_NFIP = 19.6004413492572
Finished SBL: 4142431110 in 3.25 seconds

Processing SBL: 4142431227
SBL 4142431227, Year 2031: DEU_None = -77.6896232095571, DEU_NFIP = 19.6004344413014
Finished SBL: 4142431227 in 3.42 seconds
Year 2031 processed in 26.78 seconds

Processing SBL: 3011340029
SBL 3011340029, Year 2032: DEU_None = 19.6004413493548, DEU_NFIP = 19.6004413493546
Finished SBL: 3011340029 in 3.32 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2032: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.17 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2032: DEU_None = 19.600

SBL 3011340029, Year 2038: DEU_None = 19.6004413493916, DEU_NFIP = 19.6004413493915
Finished SBL: 3011340029 in 3.71 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2038: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.18 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2038: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036250007 in 3.24 seconds

Processing SBL: 3080580018
SBL 3080580018, Year 2038: DEU_None = 19.6004413469560, DEU_NFIP = 19.6004413493488
Finished SBL: 3080580018 in 3.23 seconds

Processing SBL: 3086160111
SBL 3086160111, Year 2038: DEU_None = 19.6004413490992, DEU_NFIP = 19.6004413490990
Finished SBL: 3086160111 in 3.47 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2038: DEU_None = 19.6003764245589, DEU_NFIP = 19.6004413385631
Finished SBL: 3086320028 in 3.25 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2038: DEU_None = -649.562010415904, DEU_NFIP = 19.60041947874

SBL 3086160111, Year 2044: DEU_None = 19.6004413488423, DEU_NFIP = 19.6004413488419
Finished SBL: 3086160111 in 3.46 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2044: DEU_None = 19.6003452278261, DEU_NFIP = 19.6004413327758
Finished SBL: 3086320028 in 3.22 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2044: DEU_None = -932852.394852094, DEU_NFIP = 16.9038617005193
Finished SBL: 4142431110 in 3.19 seconds

Processing SBL: 4142431227
SBL 4142431227, Year 2044: DEU_None = -35694574.4826802, DEU_NFIP = -19516.0488098705
Finished SBL: 4142431227 in 2.94 seconds
Year 2044 processed in 26.77 seconds

Processing SBL: 3011340029
SBL 3011340029, Year 2045: DEU_None = 19.6004413494145, DEU_NFIP = 19.6004413494145
Finished SBL: 3011340029 in 3.46 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2045: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.18 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2045: DEU_None = 19.6

SBL 3011340029, Year 2051: DEU_None = 19.6004413494266, DEU_NFIP = 19.6004413494266
Finished SBL: 3011340029 in 3.58 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2051: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.57 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2051: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036250007 in 3.62 seconds

Processing SBL: 3080580018
SBL 3080580018, Year 2051: DEU_None = 19.6004413436561, DEU_NFIP = 19.6004413492890
Finished SBL: 3080580018 in 3.51 seconds

Processing SBL: 3086160111
SBL 3086160111, Year 2051: DEU_None = 19.6004413488203, DEU_NFIP = 19.6004413488196
Finished SBL: 3086160111 in 3.46 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2051: DEU_None = 19.6003006580881, DEU_NFIP = 19.6004413300222
Finished SBL: 3086320028 in 3.45 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2051: DEU_None = -10424525.1603089, DEU_NFIP = -8249.85247512

SBL 3086160111, Year 2057: DEU_None = 19.6004413485830, DEU_NFIP = 19.6004413485826
Finished SBL: 3086160111 in 3.42 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2057: DEU_None = 19.6002556187389, DEU_NFIP = 19.6004413234680
Finished SBL: 3086320028 in 3.16 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2057: DEU_None = -12447434.6742449, DEU_NFIP = -9859.00890253511
Finished SBL: 4142431110 in 2.08 seconds

Processing SBL: 4142431227
SBL 4142431227, Year 2057: DEU_None = -52418701.6210312, DEU_NFIP = -28675.5934293922
Finished SBL: 4142431227 in 1.64 seconds
Year 2057 processed in 23.35 seconds

Processing SBL: 3011340029
SBL 3011340029, Year 2058: DEU_None = 19.6004413494354, DEU_NFIP = 19.6004413494354
Finished SBL: 3011340029 in 3.60 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2058: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.48 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2058: DEU_None = 19.

SBL 3011340029, Year 2064: DEU_None = 19.6004413494408, DEU_NFIP = 19.6004413494408
Finished SBL: 3011340029 in 3.29 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2064: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.53 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2064: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036250007 in 3.20 seconds

Processing SBL: 3080580018
SBL 3080580018, Year 2064: DEU_None = 19.6004413384102, DEU_NFIP = 19.6004413492147
Finished SBL: 3080580018 in 3.45 seconds

Processing SBL: 3086160111
SBL 3086160111, Year 2064: DEU_None = 19.6004413486077, DEU_NFIP = 19.6004413486073
Finished SBL: 3086160111 in 3.20 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2064: DEU_None = 19.6001967940246, DEU_NFIP = 19.6004413211345
Finished SBL: 3086320028 in 3.17 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2064: DEU_None = -15308782.2409988, DEU_NFIP = -12130.8042818

SBL 3086160111, Year 2070: DEU_None = 19.6004413483454, DEU_NFIP = 19.6004413483447
Finished SBL: 3086160111 in 3.14 seconds

Processing SBL: 3086320028
SBL 3086320028, Year 2070: DEU_None = 19.6001403656128, DEU_NFIP = 19.6004413128533
Finished SBL: 3086320028 in 3.25 seconds

Processing SBL: 4142431110
SBL 4142431110, Year 2070: DEU_None = -18261272.3691321, DEU_NFIP = -14493.9039413340
Finished SBL: 4142431110 in 0.83 seconds

Processing SBL: 4142431227
SBL 4142431227, Year 2070: DEU_None = -56152812.2655367, DEU_NFIP = -41451.8764205404
Finished SBL: 4142431227 in 0.21 seconds
Year 2070 processed in 20.95 seconds

Processing SBL: 3011340029
SBL 3011340029, Year 2071: DEU_None = 19.6004413494450, DEU_NFIP = 19.6004413494450
Finished SBL: 3011340029 in 3.23 seconds

Processing SBL: 3036180001
SBL 3036180001, Year 2071: DEU_None = 19.6004413494696, DEU_NFIP = 19.6004413494696
Finished SBL: 3036180001 in 3.17 seconds

Processing SBL: 3036250007
SBL 3036250007, Year 2071: DEU_None = 19.

In [149]:
df.to_csv('TEST_Part_Two_245_NFIP.csv', index=False)

In [ ]:
from decimal import Decimal, getcontext
import numpy as np
import pandas as pd
import time

## Edit Test

In [148]:
tmp = tmp.sort_values(['SBL', 'Year'])
df = tmp.copy()

getcontext().prec = 15

intensity = ['00000', '00005', '00010', '00030', '00050', '00100', '00300', '00500', '01000', '10000']
weights = {k: 1 / int(k) for k in intensity if k != '00000'}
weights['00000'] = 1 - sum(weights.values())

years = np.arange(2025, 2076)
rand = np.random.randint(0, 10001, size=years.size)

conds = [
    rand < 6303,
    rand < 8303,
    rand < 9303,
    rand < 9636,
    rand < 9836,
    rand < 9936,
    rand < 9969,
    rand < 9989,
    rand < 9999,
]
choices = [0, 5, 10, 30, 50, 100, 300, 500, 1000]
flood_vals = np.select(conds, choices, default=10000)
year_to_event = dict(zip(years, flood_vals))

df['Flood_Event'] = df['Year'].map(year_to_event)

# --- Main loop ---
start_time = time.time()

for year in years:
    year_start = time.time()

    for sbl in df.loc[df['Year']==year, 'SBL'].unique():
        sbl_start = time.time()
        print(f"\nProcessing SBL: {sbl}")

        # mask for this parcel & year
        m_cur = (df['SBL']==sbl) & (df['Year']==year)
        base = df.loc[m_cur]
        if base.empty:
            continue

        occupy = base['Occupy_NFIP'].iat[0]
        if occupy == 0:
            zero_cols = [
                'Dam_T','Dam_T_Con','Dam_T_Struc',
                'Dam_NFIP','Dam_NFIP_Con','Dam_NFIP_Struc','Dam_NFIP_Deduct',
                'Dam_NFIP_Private','Dam_NFIP_Private_Deduct','Dam_None',
                'NFIP_Premium_Paid','NFIP_Private_Premium_Paid',
                'Decision_NFIP','Decision_None'
            ]
            df.loc[m_cur, zero_cols] = 0
            continue

        # compute base factors
        flood_occur_min = min(
            df.loc[m_cur, f'Flood_Occur_{i}_Factor'].fillna(1).iat[0]
            for i in range(1,5)
        )
        prox_flood_min = min(
            df.loc[m_cur, f'Prox_Flood_{i}_Factor'].fillna(1).iat[0]
            for i in range(1,5)
        )

        deu_none = Decimal('0')
        deu_nfip = Decimal('0')

        slr_year = df.loc[df['SBL']==sbl, 'SLR'].dropna().iat[0]
        end_year = min(year+30, slr_year+1)

        # loop intensities
        for suffix in intensity:
            npv_none = []
            npv_nfip  = []

            # update factors for each future year y
            for t, y in enumerate(range(year, end_year), start=1):
                m_y = (df['SBL']==sbl) & (df['Year']==y)

                # Flood/Prox factor
                if suffix == '00000':
                    fv = flood_occur_min
                    pv = prox_flood_min
                else:
                    col0  = f'Flood_Occur_0_{suffix}_Factor'
                    colp0 = f'Prox_Flood_0_{suffix}_Factor'
                    raw = df.loc[m_y, col0].fillna(1).values
                    if raw.size:
                        fv = min(flood_occur_min, raw[0])
                        pv = min(prox_flood_min,
                                 df.loc[m_y, colp0].fillna(1).iat[0])
                    else:
                        fv = 1
                        pv = 1
                df.loc[m_y, f'Flood_Occur_{suffix}_Factor'] = fv
                df.loc[m_y, f'Prox_Flood_{suffix}_Factor']  = pv
                df.loc[m_y, f'Flood_Factor_{suffix}']       = min(fv, pv)

                # project market value
                iv = df.loc[m_y, 'Initial_Market_Value'].fillna(1).iat[0]
                ev = df.loc[m_y, 'External_Factor'].fillna(1).iat[0]
                sv = df.loc[m_y, 'SLR_Factor'].fillna(1).iat[0]
                proj = iv * ev * sv * min(fv, pv)
                df.loc[m_y, f'Market_Project_{suffix}'] = proj

                # compute NPVs
                mv = Decimal(str(proj))
                if suffix == '00000':
                    dt = Decimal('0')
                    dd = Decimal('0')
                    dp = Decimal('0')
                else:
                    dt = Decimal(str(df.loc[m_y, f'Dam_T_{suffix}'].fillna(0).iat[0]))
                    dd = Decimal(str(df.loc[m_y, f'Dam_NFIP_Deduct_{suffix}'].fillna(0).iat[0]))
                    dp = Decimal(str(df.loc[m_y, f'Dam_NFIP_Private_Deduct_{suffix}'].fillna(0).iat[0]))

                pn = Decimal(str(df.loc[m_y, 'NFIP_Premium'].fillna(0).iat[0]))
                pp = Decimal(str(df.loc[m_y, 'NFIP_Private_Premium'].fillna(0).iat[0]))
                rc = Decimal(str(df.loc[m_y, 'ReplaceCost'].fillna(0).iat[0]))

                if rc == 0:
                    npv_n = Decimal('0')
                    npv_f = Decimal('0')
                else:
                    exp_n = (-1/(Decimal('0.05')*rc))*(mv - dt)
                    exp_f = (-1/(Decimal('0.05')*rc))*(mv - dd - dp - pn - pp)
                    npv_n = (1/(Decimal('1.03')**t))*(1 - exp_n.exp())
                    npv_f = (1/(Decimal('1.03')**t))*(1 - exp_f.exp())

                npv_none.append(npv_n)
                npv_nfip.append(npv_f)

            w = Decimal(str(weights[suffix]))
            deu_none += w * sum(npv_none)
            deu_nfip  += w * sum(npv_nfip)

        # write DEU and decisions
        df.loc[m_cur, 'DEU_None']   = deu_none
        df.loc[m_cur, 'DEU_NFIP']   = deu_nfip
        df.loc[m_cur, 'Decision_None'] = int(deu_none >= deu_nfip)
        df.loc[m_cur, 'Decision_NFIP'] = int(deu_none < deu_nfip)

        print(f"SBL {sbl}, Year {year}: DEU_None={deu_none}, DEU_NFIP={deu_nfip}")

        # --- damage & factor assignment (replicating Code A logic) ---
        # zero‐event
        zm = m_cur & (df['Flood_Event']==0)
        df.loc[zm, ['Dam_T','Dam_T_Con','Dam_T_Struc']] = 0
        df.loc[zm, 'Flood_Occur_0_Factor'] = 1
        df.loc[zm, 'Flood_Occur_Factor']   = df.loc[zm, 'Flood_Occur_00000_Factor']
        df.loc[zm, 'Prox_Flood_0_Factor']  = 1
        df.loc[zm, 'Prox_Flood_Factor']    = df.loc[zm, 'Prox_Flood_00000_Factor']
        df.loc[zm, 'Flood_Factor']         = df.loc[zm, 'Flood_Factor_00000']
        df.loc[zm, 'Market_Value_Avg_NFIP'] = df.loc[zm, 'Market_Project_00000']
        df.loc[zm, ['Dam_NFIP_Con','Dam_NFIP_Struc','Dam_NFIP',
                    'Dam_NFIP_Deduct','Dam_NFIP_Private','Dam_NFIP_Private_Deduct',
                    'Dam_None']] = 0

        # premiums paid
        df.loc[zm & (df['Decision_NFIP']==1), 'NFIP_Premium_Paid'] = df.loc[zm, 'NFIP_Premium']
        df.loc[zm & (df['Decision_NFIP']==1), 'NFIP_Private_Premium_Paid'] = df.loc[zm, 'NFIP_Private_Premium']
        df.loc[zm & (df['Decision_None']==1), ['NFIP_Premium_Paid','NFIP_Private_Premium_Paid']] = 0

        # non‐zero events
        for suffix in intensity:
            if suffix == '00000':
                continue
            ev = int(suffix)
            fm = m_cur & (df['Flood_Event']==ev)

            # structural & contents damage
            df.loc[fm, 'Dam_T_Con'] = df.loc[fm, f'Dam_T_Con_{suffix}']
            df.loc[fm, 'Dam_T_Struc'] = df.loc[fm, f'Dam_T_Struc_{suffix}']
            df.loc[fm, 'Dam_T']       = df.loc[fm, f'Dam_T_{suffix}']

            # factors
            df.loc[fm, 'Flood_Occur_0_Factor'] = df.loc[fm, f'Flood_Occur_0_{suffix}_Factor']
            df.loc[fm, 'Flood_Occur_Factor']   = df.loc[fm, f'Flood_Occur_{suffix}_Factor']
            df.loc[fm, 'Prox_Flood_0_Factor']  = df.loc[fm, f'Prox_Flood_0_{suffix}_Factor']
            df.loc[fm, 'Prox_Flood_Factor']    = df.loc[fm, f'Prox_Flood_{suffix}_Factor']
            df.loc[fm, 'Flood_Factor']         = df.loc[fm, f'Flood_Factor_{suffix}']

            # average market value under NFIP
            df.loc[fm, 'Market_Value_Avg_NFIP'] = df.loc[fm, f'Market_Project_{suffix}']

            # NFIP damage splits & premiums
            nm = fm & (df['Decision_NFIP']==1)
            df.loc[nm, 'Dam_NFIP_Con'] = df.loc[nm, f'Dam_NFIP_Con_{suffix}']
            df.loc[nm, 'Dam_NFIP_Struc']= df.loc[nm, f'Dam_NFIP_Struc_{suffix}']
            df.loc[nm, 'Dam_NFIP']      = df.loc[nm, f'Dam_NFIP_{suffix}']
            df.loc[nm, 'Dam_NFIP_Deduct'] = df.loc[nm, f'Dam_NFIP_Deduct_{suffix}']
            df.loc[nm, 'Dam_NFIP_Private'] = df.loc[nm, f'Dam_NFIP_Private_{suffix}']
            df.loc[nm, 'Dam_NFIP_Private_Deduct'] = df.loc[nm, f'Dam_NFIP_Private_Deduct_{suffix}']
            df.loc[nm, 'Dam_None']      = 0
            df.loc[nm, 'NFIP_Premium_Paid'] = df.loc[nm, 'NFIP_Premium']
            df.loc[nm, 'NFIP_Private_Premium_Paid'] = df.loc[nm, 'NFIP_Private_Premium']

            nn = fm & (df['Decision_None']==1)
            df.loc[nn, ['Dam_NFIP_Con','Dam_NFIP_Struc','Dam_NFIP',
                        'Dam_NFIP_Deduct','Dam_NFIP_Private','Dam_NFIP_Private_Deduct']] = 0
            df.loc[nn, 'Dam_None'] = df.loc[nn, 'Dam_T']
            df.loc[nn, ['NFIP_Premium_Paid','NFIP_Private_Premium_Paid']] = 0

        print(f"Finished SBL: {sbl} in {time.time()-sbl_start:.2f}s")

    # --- cumulative sums only once per year ---
    mask_year = df['Year'] <= year
    df.loc[mask_year, 'Cost_NFIP']                   = df.loc[mask_year].groupby('SBL')['Dam_NFIP'].cumsum()
    df.loc[mask_year, 'Cost_NFIP_Deduct']            = df.loc[mask_year].groupby('SBL')['Dam_NFIP_Deduct'].cumsum()
    df.loc[mask_year, 'Cost_NFIP_Private']           = df.loc[mask_year].groupby('SBL')['Dam_NFIP_Private'].cumsum()
    df.loc[mask_year, 'Cost_NFIP_Private_Deduct']    = df.loc[mask_year].groupby('SBL')['Dam_NFIP_Private_Deduct'].cumsum()
    df.loc[mask_year, 'Cost_None']                   = df.loc[mask_year].groupby('SBL')['Dam_None'].cumsum()
    df.loc[mask_year, 'Cost_NFIP_Premium']           = df.loc[mask_year].groupby('SBL')['NFIP_Premium_Paid'].cumsum()
    df.loc[mask_year, 'Cost_NFIP_Private_Premium']   = df.loc[mask_year].groupby('SBL')['NFIP_Private_Premium_Paid'].cumsum()

    print(f"Year {year} processed in {time.time()-year_start:.2f}s")

total_duration = time.time() - start_time
print(f"\nTotal time for all SBLs and years: {total_duration:.2f}s")


Processing SBL: 3011340029
SBL 3011340029, Year 2025: DEU_None=19.6004413492429, DEU_NFIP=19.6004413492427
Finished SBL: 3011340029 in 2.03s

Processing SBL: 3036180001
SBL 3036180001, Year 2025: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 2.23s

Processing SBL: 3036250007
SBL 3036250007, Year 2025: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 2.05s

Processing SBL: 3080580018
SBL 3080580018, Year 2025: DEU_None=19.6004413484797, DEU_NFIP=19.6004413493829
Finished SBL: 3080580018 in 1.95s

Processing SBL: 3086160111
SBL 3086160111, Year 2025: DEU_None=19.6004413492480, DEU_NFIP=19.6004413492479
Finished SBL: 3086160111 in 1.93s

Processing SBL: 3086320028
SBL 3086320028, Year 2025: DEU_None=19.6004189104397, DEU_NFIP=19.6004413445551
Finished SBL: 3086320028 in 1.95s

Processing SBL: 4142431110
SBL 4142431110, Year 2025: DEU_None=19.5975512392253, DEU_NFIP=19.6004413493562
Finished SBL: 4142431110 in 1.83s

Proce

SBL 3036180001, Year 2032: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 1.98s

Processing SBL: 3036250007
SBL 3036250007, Year 2032: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 1.91s

Processing SBL: 3080580018
SBL 3080580018, Year 2032: DEU_None=19.6004413478406, DEU_NFIP=19.6004413493736
Finished SBL: 3080580018 in 1.94s

Processing SBL: 3086160111
SBL 3086160111, Year 2032: DEU_None=19.6004413492387, DEU_NFIP=19.6004413492383
Finished SBL: 3086160111 in 1.93s

Processing SBL: 3086320028
SBL 3086320028, Year 2032: DEU_None=19.6004003870269, DEU_NFIP=19.6004413418283
Finished SBL: 3086320028 in 1.98s

Processing SBL: 4142431110
SBL 4142431110, Year 2032: DEU_None=19.5922356042461, DEU_NFIP=19.6004413492958
Finished SBL: 4142431110 in 1.92s

Processing SBL: 4142431227
SBL 4142431227, Year 2032: DEU_None=-394.205943492880, DEU_NFIP=19.6004235603664
Finished SBL: 4142431227 in 2.02s
Year 2032 processed in 15.65s

Pr

SBL 3036250007, Year 2039: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 1.95s

Processing SBL: 3080580018
SBL 3080580018, Year 2039: DEU_None=19.6004413467755, DEU_NFIP=19.6004413493492
Finished SBL: 3080580018 in 1.92s

Processing SBL: 3086160111
SBL 3086160111, Year 2039: DEU_None=19.6004413491568, DEU_NFIP=19.6004413491566
Finished SBL: 3086160111 in 1.97s

Processing SBL: 3086320028
SBL 3086320028, Year 2039: DEU_None=19.6003716579545, DEU_NFIP=19.6004413381462
Finished SBL: 3086320028 in 2.17s

Processing SBL: 4142431110
SBL 4142431110, Year 2039: DEU_None=-2378.37576358910, DEU_NFIP=19.6003734036330
Finished SBL: 4142431110 in 1.96s

Processing SBL: 4142431227
SBL 4142431227, Year 2039: DEU_None=-2543886.87905240, DEU_NFIP=14.1957390756554
Finished SBL: 4142431227 in 2.20s
Year 2039 processed in 16.16s

Processing SBL: 3011340029
SBL 3011340029, Year 2040: DEU_None=19.6004413493997, DEU_NFIP=19.6004413493996
Finished SBL: 3011340029 in 2.01s

P

SBL 3080580018, Year 2046: DEU_None=19.6004413452021, DEU_NFIP=19.6004413493250
Finished SBL: 3080580018 in 2.10s

Processing SBL: 3086160111
SBL 3086160111, Year 2046: DEU_None=19.6004413490720, DEU_NFIP=19.6004413490718
Finished SBL: 3086160111 in 2.06s

Processing SBL: 3086320028
SBL 3086320028, Year 2046: DEU_None=19.6003334142093, DEU_NFIP=19.6004413340094
Finished SBL: 3086320028 in 1.93s

Processing SBL: 4142431110
SBL 4142431110, Year 2046: DEU_None=-8992282.52421032, DEU_NFIP=-7108.08805428644
Finished SBL: 4142431110 in 1.95s

Processing SBL: 4142431227
SBL 4142431227, Year 2046: DEU_None=-37868335.3258042, DEU_NFIP=-20667.3702129071
Finished SBL: 4142431227 in 1.73s
Year 2046 processed in 16.03s

Processing SBL: 3011340029
SBL 3011340029, Year 2047: DEU_None=19.6004413494193, DEU_NFIP=19.6004413494193
Finished SBL: 3011340029 in 2.16s

Processing SBL: 3036180001
SBL 3036180001, Year 2047: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 2.00s


Finished SBL: 3080580018 in 2.08s

Processing SBL: 3086160111
SBL 3086160111, Year 2053: DEU_None=19.6004413489875, DEU_NFIP=19.6004413489871
Finished SBL: 3086160111 in 2.08s

Processing SBL: 3086320028
SBL 3086320028, Year 2053: DEU_None=19.6002863333619, DEU_NFIP=19.6004413297409
Finished SBL: 3086320028 in 2.14s

Processing SBL: 4142431110
SBL 4142431110, Year 2053: DEU_None=-11059380.9261326, DEU_NFIP=-8749.71419328636
Finished SBL: 4142431110 in 1.69s

Processing SBL: 4142431227
SBL 4142431227, Year 2053: DEU_None=-46573283.6018155, DEU_NFIP=-25425.9209540950
Finished SBL: 4142431227 in 1.36s
Year 2053 processed in 15.84s

Processing SBL: 3011340029
SBL 3011340029, Year 2054: DEU_None=19.6004413494308, DEU_NFIP=19.6004413494308
Finished SBL: 3011340029 in 2.00s

Processing SBL: 3036180001
SBL 3036180001, Year 2054: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 2.04s

Processing SBL: 3036250007
SBL 3036250007, Year 2054: DEU_None=19.6004413494696

SBL 3086160111, Year 2060: DEU_None=19.6004413489055, DEU_NFIP=19.6004413489053
Finished SBL: 3086160111 in 2.32s

Processing SBL: 3086320028
SBL 3086320028, Year 2060: DEU_None=19.6002308362794, DEU_NFIP=19.6004413253138
Finished SBL: 3086320028 in 2.17s

Processing SBL: 4142431110
SBL 4142431110, Year 2060: DEU_None=-13601651.2222010, DEU_NFIP=-10768.7072784406
Finished SBL: 4142431110 in 1.22s

Processing SBL: 4142431227
SBL 4142431227, Year 2060: DEU_None=-57279271.9795629, DEU_NFIP=-31278.3381479801
Finished SBL: 4142431227 in 0.84s
Year 2060 processed in 14.66s

Processing SBL: 3011340029
SBL 3011340029, Year 2061: DEU_None=19.6004413494383, DEU_NFIP=19.6004413494383
Finished SBL: 3011340029 in 1.94s

Processing SBL: 3036180001
SBL 3036180001, Year 2061: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 2.01s

Processing SBL: 3036250007
SBL 3036250007, Year 2061: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 2.20s


Finished SBL: 3086160111 in 2.14s

Processing SBL: 3086320028
SBL 3086320028, Year 2067: DEU_None=19.6001695685479, DEU_NFIP=19.6004413203577
Finished SBL: 3086320028 in 2.09s

Processing SBL: 4142431110
SBL 4142431110, Year 2067: DEU_None=-16727816.1404789, DEU_NFIP=-13251.8140959043
Finished SBL: 4142431110 in 0.74s

Processing SBL: 4142431227
SBL 4142431227, Year 2067: DEU_None=-69934500.1207931, DEU_NFIP=-38476.0199774144
Finished SBL: 4142431227 in 0.45s
Year 2067 processed in 13.19s

Processing SBL: 3011340029
SBL 3011340029, Year 2068: DEU_None=19.6004413494434, DEU_NFIP=19.6004413494434
Finished SBL: 3011340029 in 1.96s

Processing SBL: 3036180001
SBL 3036180001, Year 2068: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 1.95s

Processing SBL: 3036250007
SBL 3036250007, Year 2068: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 1.95s

Processing SBL: 3080580018
SBL 3080580018, Year 2068: DEU_None=19.6004413360695

SBL 3011340029, Year 2075: DEU_None=19.6004413494469, DEU_NFIP=19.6004413494469
Finished SBL: 3011340029 in 1.94s

Processing SBL: 3036180001
SBL 3036180001, Year 2075: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036180001 in 1.91s

Processing SBL: 3036250007
SBL 3036250007, Year 2075: DEU_None=19.6004413494696, DEU_NFIP=19.6004413494696
Finished SBL: 3036250007 in 1.99s

Processing SBL: 3080580018
SBL 3080580018, Year 2075: DEU_None=19.6004413306866, DEU_NFIP=19.6004413491342
Finished SBL: 3080580018 in 1.92s

Processing SBL: 3086160111
SBL 3086160111, Year 2075: DEU_None=19.6004413487461, DEU_NFIP=19.6004413487458
Finished SBL: 3086160111 in 1.94s

Processing SBL: 3086320028
SBL 3086320028, Year 2075: DEU_None=19.6000831629557, DEU_NFIP=19.6004413146922
Finished SBL: 3086320028 in 1.99s

Processing SBL: 4142431110
SBL 4142431110, Year 2075: DEU_None=-14030666.7189843, DEU_NFIP=-16530.8923266104
Finished SBL: 4142431110 in 0.26s

Processing SBL: 4142431227
Year

## 585 Set Up

# FloResVAP Set Up

## 245 Set Up

## 585 Set Up

# Work

In [6]:
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00005'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00010'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00030'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00050'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00100'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00300'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_00500'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5    
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_01000'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Con_10000'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost'] * 0.5
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00005'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00010'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00030'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00050'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00100'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00300'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_00500'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']    
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_01000'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_Struc_10000'] *
    Result_245_Part_Two_Shell_NFIP_FIRM['ReplaceCost']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00005'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00005']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00010'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00010']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00030'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00030']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00050'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00050']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00100'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00100']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00300'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00300']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00500'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00500']    
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_01000'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_01000']
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_10000'] +
    Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_10000']
    
conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00005'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00005'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00005']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00005'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00010'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00010'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00010']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00010'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00030'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00030'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00030']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00030'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00050'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00050'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00050']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00050'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00100'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00100'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00100']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00100'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00300'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00300'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00300']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00300'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00500'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00500'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_00500']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00500'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_01000'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_01000'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_01000']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_01000'] = np.select(conditions, choices, default=default)

conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_10000'] >= 111111.11),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_10000'] < 111111.11)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Con_10000']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_10000'] = np.select(conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00005'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00005'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00005']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00005'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00010'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00010'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00010']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00010'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00030'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00030'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00030']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00030'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00050'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00050'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00050']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00050'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00100'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00100'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00100']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00100'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00300'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00300'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00300']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00300'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00500'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00500'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_00500']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00500'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_01000'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_01000'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_01000']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_01000'] = np.select(Conditions, choices, default=default)

Conditions = [
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_10000'] >= 277777.78),
        (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_10000'] < 277777.78)
    ]

choices = [
    100000,
    0.9 * Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_Struc_10000']
]

default = 0

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_10000'] = np.select(Conditions, choices, default=default)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00005'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00005']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00010'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00010']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00030'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00030']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00050'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00050']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00100'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00100']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00300'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00300']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00500'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00500']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_00100'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_00100']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_01000'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_01000']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Con_10000'] + Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Struc_10000']

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00005'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00010'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00030'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00050'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00100'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00300'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00500'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_01000'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_10000'] * (1/9)
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00005'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00005'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00005'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00005']) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00010'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00010'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00010'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00010'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00030'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00030'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00030'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00030'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00050'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00050'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00050'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00050'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00100'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00100'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00100'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00100'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00300'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00300'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00300'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00300'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00500'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00500'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_00500'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_00500'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_01000'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_01000'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_01000'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_01000'])

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_10000'] = 0.9 * (Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_10000'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_10000'] - Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Deduct_10000'])
    
Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00005'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00005'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00010'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00010'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00030'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00030'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00050'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00050'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00100'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00100'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00300'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00300'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_00500'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_00500'] * (1/9) 

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_01000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_01000'] * (1/9)

Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_Deduct_10000'] = Result_245_Part_Two_Shell_NFIP_FIRM['Dam_NFIP_Private_10000'] * (1/9)
    
Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00005_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00005'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00010_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00010'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00030_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00030'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00050_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00050'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00100_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00100'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00300_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00300'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_00500_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_00500'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_01000_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_01000'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Flood_Occur_0_10000_Factor'] = np.where(
        Result_245_Part_Two_Shell_NFIP_FIRM['Dam_T_10000'] > 0,
        0.805,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00005_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00005'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00010_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00010'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00030_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00030'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00050_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00050'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00100_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00100'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00300_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00300'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_00500_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_00500'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_01000_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_01000'] == 'Y'),
        0.935,
        0
    )

Result_245_Part_Two_Shell_NFIP_FIRM['Prox_Flood_0_10000_Factor'] = np.where(
        (Result_245_Part_Two_Shell_NFIP_FIRM['Prox_10000'] == 'Y'),
        0.935,
        0
    )

df['Occupy_NFIP'] = np.where(df['SLR'] < df['Year'], 0, 1)


SyntaxError: invalid syntax (2891724279.py, line 1)

In [ ]:
def simulate_floresvap(seed, Result_245, Proximity_164, Proximity_197, Proximity_328, verbose=False, save_path=None):
    if verbose:
        print(f"  -> Running seed {seed}")

    np.random.seed(seed)
    df = Result_245.copy()

    for Year in range(2025, 2106):
        random_number = np.random.randint(0, 10001)
   
        if random_number < 6303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 0
        elif 6303 <= random_number < 8303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 5
        elif 8303 <= random_number < 9303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 10
        elif 9303 <= random_number < 9636:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 30
        elif 9636 <= random_number < 9836:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 50
        elif 9836 <= random_number < 9936:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 100
        elif 9936 <= random_number < 9969:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 300
        elif 9969 <= random_number < 9989:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 500
        elif 9989 <= random_number < 9999:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 1000
        else:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 10000
       

    df['Occupy_NFIP'] = np.where(df['SLR'] < df['Year'], 0, 1)

    conditions = [
        (df['SLR'] == df['Year']),
        (df['Flood_Event'] == 0) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 5) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 10) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 30) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 50) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 100) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 300) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 500) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 1000) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 10000) & (df['SLR'] != df['Year']),
    ]

    choices_struc = [
        df['ReplaceCost'],
        0,
        df['ReplaceCost'] * df['Dam_Struc_00005'],
        df['ReplaceCost'] * df['Dam_Struc_00010'],
        df['ReplaceCost'] * df['Dam_Struc_00030'],
        df['ReplaceCost'] * df['Dam_Struc_00050'],
        df['ReplaceCost'] * df['Dam_Struc_00100'],
        df['ReplaceCost'] * df['Dam_Struc_00300'],
        df['ReplaceCost'] * df['Dam_Struc_00500'],
        df['ReplaceCost'] * df['Dam_Struc_01000'],
        df['ReplaceCost'] * df['Dam_Struc_10000'],
    ]

    df['Dam_T_Struc'] = np.select(conditions, choices_struc, default=np.nan)

    choices_con = [
        df['ReplaceCost'] * 0.5,
        0,
        df['ReplaceCost'] * df['Dam_Con_00005'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00010'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00030'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00050'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00100'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00300'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00500'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_01000'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_10000'] * 0.5,
    ]

    df['Dam_T_Con'] = np.select(conditions, choices_con, default=np.nan)

    df['Dam_T'] = df['Dam_T_Con'] + df['Dam_T_Struc']

    df['Dam_T_NFIP'] = np.where(
        df['Occupy_NFIP'] == 1,
        df['Dam_T'],
        0
    )

    conditions = [
        (df['Occupy_NFIP'] == 1) & (df['Dam_T_Struc'] >= 277777.78),
        (df['Occupy_NFIP'] == 1) & (df['Dam_T_Struc'] < 277777.78)
    ]

    choices = [
        250000,
        0.9 * df['Dam_T_Struc']
    ]

    default = 0

    df['Dam_NFIP_Struc'] = np.select(conditions, choices, default=default)

    conditions = [
        (df['Occupy_NFIP'] == 1) & (df['Dam_T_Con'] >= 111111.11),
        (df['Occupy_NFIP'] == 1) & (df['Dam_T_Con'] < 111111.11)
    ]

    choices = [
        100000,
        0.9 * df['Dam_T_Con']
    ]

    default = 0

    df['Dam_NFIP_Con'] = np.select(conditions, choices, default=default)

    df['Dam_NFIP'] = df['Dam_NFIP_Con'] + df['Dam_NFIP_Struc']

    df['Dam_NFIP_Deduct_Struc'] = df['Dam_NFIP_Struc'] * (1/9)

    df['Dam_NFIP_Deduct_Con'] = df['Dam_NFIP_Con'] * (1/9)

    df['Dam_NFIP_Deduct'] = df['Dam_NFIP_Deduct_Con'] + df['Dam_NFIP_Deduct_Struc']

    df['Dam_NFIP_Private_Struc'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        (df['Dam_T_Struc'] - df['Dam_NFIP_Struc'] - df['Dam_NFIP_Deduct_Struc']) * 0.9
    )

   
    df['Dam_NFIP_Private_Con'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        (df['Dam_T_Con'] - df['Dam_NFIP_Con'] - df['Dam_NFIP_Deduct_Con']) * 0.9
    )


    df['Dam_NFIP_Private'] = df['Dam_NFIP_Private_Con'] + df['Dam_NFIP_Private_Struc']


    df['Dam_NFIP_Private_Deduct_Struc'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        df['Dam_NFIP_Private_Struc'] * (1/9)
    )

    df['Dam_NFIP_Private_Deduct_Con'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        df['Dam_NFIP_Private_Con'] * (1/9)
    )


    df['Dam_NFIP_Private_Deduct'] = df['Dam_NFIP_Private_Deduct_Con'] + df['Dam_NFIP_Private_Deduct_Struc']


    df['Flood_Occur_0_Factor'] = np.where(df['Dam_T'] > 0, 0.805, 1)

    df = df.sort_values(by=['SBL', 'Year'])

    df['Dam_T_prev'] = df.groupby('SBL')['Dam_T'].shift(1)

    df['Flood_Occur_1_Factor'] = np.where(df['Dam_T_prev'] > 0, 0.844, 1)

    df = df.drop(columns='Dam_T_prev')


    df['Dam_T_prev2'] = df.groupby('SBL')['Dam_T'].shift(2)

    df['Flood_Occur_2_Factor'] = np.where(df['Dam_T_prev2'] > 0, 0.883, 1)

    df = df.drop(columns='Dam_T_prev2')


    df['Dam_T_prev3'] = df.groupby('SBL')['Dam_T'].shift(3)

    df['Flood_Occur_3_Factor'] = np.where(df['Dam_T_prev3'] > 0, 0.922, 1)

    df = df.drop(columns='Dam_T_prev3')


    df['Dam_T_prev4'] = df.groupby('SBL')['Dam_T'].shift(4)

    df['Flood_Occur_4_Factor'] = np.where(df['Dam_T_prev4'] > 0, 0.961, 1)

    df = df.drop(columns='Dam_T_prev4')

    flood_cols = [
        'Flood_Occur_0_Factor',
        'Flood_Occur_1_Factor',
        'Flood_Occur_2_Factor',
        'Flood_Occur_3_Factor',
        'Flood_Occur_4_Factor'
    ]

    df['Flood_Occur_Factor'] = df[flood_cols].min(axis=1)


    conditions = [
        (df['Flood_Event'] == 0),
        (df['Flood_Event'] == 5) & (df['Prox_00005'] == 'Y'),
        (df['Flood_Event'] == 10) & (df['Prox_00010'] == 'Y'),
        (df['Flood_Event'] == 30) & (df['Prox_00030'] == 'Y'),
        (df['Flood_Event'] == 50) & (df['Prox_00050'] == 'Y'),
        (df['Flood_Event'] == 100) & (df['Prox_00100'] == 'Y'),
        (df['Flood_Event'] == 300) & (df['Prox_00300'] == 'Y'),
        (df['Flood_Event'] == 500) & (df['Prox_00500'] == 'Y'),
        (df['Flood_Event'] == 1000) & (df['Prox_01000'] == 'Y'),
        (df['Flood_Event'] == 10000) & (df['Prox_10000'] == 'Y')
    ]

    choices = [
        1,        
        0.935,  
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935
    ]

    default = 1

    df['Prox_Flood_0_Factor'] = np.select(conditions, choices, default=default)


    df = df.sort_values(by=['SBL', 'Year'])

    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(1)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(1)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(1)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(1)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(1)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(1)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(1)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(1)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(1)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(1)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.948,    
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
    ]

    df['Prox_Flood_1_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(2)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(2)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(2)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(2)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(2)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(2)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(2)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(2)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(2)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(2)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.961,    
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
    ]

    df['Prox_Flood_2_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(3)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(3)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(3)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(3)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(3)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(3)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(3)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(3)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(3)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(3)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.974,    
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
    ]

    df['Prox_Flood_3_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])

    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(4)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(4)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(4)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(4)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(4)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(4)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(4)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(4)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(4)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(4)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.987,    
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
    ]

    df['Prox_Flood_4_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    flood_cols = [
        'Prox_Flood_0_Factor',
        'Prox_Flood_1_Factor',
        'Prox_Flood_2_Factor',
        'Prox_Flood_3_Factor',
        'Prox_Flood_4_Factor'
    ]

    df['Prox_Flood_Factor'] = df[flood_cols].min(axis=1)


    df['Flood_Factor'] = df[['Flood_Occur_Factor', 'Prox_Flood_Factor']].min(axis=1)

    df['Market_Value_Avg_NFIP'] = np.where(
        df['Occupy_NFIP'] == 0,
        np.nan,
        df['Initial_Market_Value'] *
        df['External_Factor'] *
        df['SLR_Factor'] *
        df['Flood_Factor']
    )


    df = df.sort_values(by=['SBL', 'Year'])
    grouped = df.groupby('SBL')
   
    df['Cost_NFIP'] = grouped['Dam_NFIP'].cumsum()


    df['Cost_NFIP_Deduct'] = grouped['Dam_NFIP_Deduct'].cumsum()


    df['Cost_NFIP_Private'] = grouped['Dam_NFIP_Private'].cumsum()


    df['Cost_NFIP_Private_Deduct'] = grouped['Dam_NFIP_Private_Deduct'].cumsum()

    for year in range(2025, 2106):

   
        df['Occupy_Alt'] = np.where(
            df.groupby('SBL')['Occupy_Alt'].shift(1).fillna(1) == 0, 0,
            np.where(df['SLR'] < df["Year"], 0,
            np.where(df.groupby('SBL')['Alt_Buyout'].shift(1).fillna(0) > 0, 0, 1))
        )


        df['Dam_T_Alt'] = np.where(
            df['Occupy_Alt'] == 1,
            df['Dam_T'],
            0
        )
   
   
        df['Dam_Alt'] = np.where(
            (df['Occupy_Alt'] == 1) &
            (df['SLR'] != df["Year"]) &
            (df['Dam_T_Struc'] + df.groupby('SBL')['Cost_Alt'].shift(1).fillna(0) <= 0.1 * df['ReplaceCost']),
            df['Dam_T_Struc'],
            0
        )
   
       
        df['Dam_Alt_Private'] = np.where(
            (df['Occupy_Alt'] == 1) & (df['Dam_T_Con'] > 0),
            df['Dam_T_Con'] * 0.9,
            0
        )
   
   
        df['Dam_Alt_Private_Deduct'] = np.where(
            (df['Occupy_Alt'] == 1) & (df['Dam_T_Con'] > 0),
            df['Dam_T_Con'] * 0.1,
            0
        )
   
       
        df['Alt_Buyout'] = np.where(
            (df['Occupy_Alt'] == 1) &
            (df['Mean_Forecast'] >= 1) &
            ((df['Dam_T_Struc'] + df.groupby('SBL')['Cost_Alt'].shift(1).fillna(0)) > 0.1 * df['ReplaceCost']),
       
            df['Mean_Forecast'] * df['Market_Value_2024'],
       
            np.where(
                (df['Occupy_Alt'] == 1) &
                (df['Mean_Forecast'] < 1) &
                ((df['Dam_T_Struc'] + df.groupby('SBL')['Cost_Alt'].shift(1).fillna(0)) > 0.1 * df['ReplaceCost']),
           
                df['Market_Value_2024'],
           
            0)
        )
   
       
        conditions_164 = [
            df['Year'] == 2024,
            df['Year'] < year,
            df['SBL'].isin(Proximity_164.loc[
                Proximity_164['SBL_x'].isin(
                    df.loc[(df['Year'] == year) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_164.loc[
                Proximity_164['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 1)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_164.loc[
                Proximity_164['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_164.loc[
                Proximity_164['SBL_x'].isin(
                    df.loc[(df['Year'] < (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['Year'] == year
        ]
   
        choices_164 = [1, df['Proximity_164_Factor'], 0.794, 0.863, 0.931, 1.069, 1]
   
        df['Proximity_164_Factor'] = np.select(conditions_164, choices_164, default=df['Proximity_164_Factor'])

       
        conditions_197 = [
            df['Year'] == 2024,
            df['Year'] < year,
            df['SBL'].isin(Proximity_197.loc[
                Proximity_197['SBL_x'].isin(
                    df.loc[(df['Year'] == year) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_197.loc[
                Proximity_197['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 1)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_197.loc[
                Proximity_197['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_197.loc[
                Proximity_197['SBL_x'].isin(
                    df.loc[(df['Year'] < (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['Year'] == year
        ]

        choices_197 = [1, df['Proximity_197_Factor'], 0.837, 0.891, 0.946, 1.054, 1]
   
        df['Proximity_197_Factor'] = np.select(conditions_197, choices_197, default=df['Proximity_197_Factor'])
   
       
        conditions_328 = [
            df['Year'] == 2024,
            df['Year'] < year,
            df['SBL'].isin(Proximity_328.loc[
                Proximity_328['SBL_x'].isin(
                    df.loc[(df['Year'] == year) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_328.loc[
                Proximity_328['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 1)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_328.loc[
                Proximity_328['SBL_x'].isin(
                    df.loc[(df['Year'] == (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['SBL'].isin(Proximity_328.loc[
                Proximity_328['SBL_x'].isin(
                    df.loc[(df['Year'] < (year - 2)) & (df['Alt_Buyout'] > 0), 'SBL']
                ), 'SBL_y'
            ]),
            df['Year'] == year
        ]
   
        choices_328 = [1, df['Proximity_328_Factor'], 0.871, 0.914, 0.957, 1.043, 1]
   
        df['Proximity_328_Factor'] = np.select(conditions_328, choices_328, default=df['Proximity_328_Factor'])

       
        condition_buyout = (
            ((df['Proximity_164_Factor'] > 1) | (df['Proximity_197_Factor'] > 1) | (df['Proximity_328_Factor'] > 1)) &
            (df['Proximity_164_Factor'] >= 1) & (df['Proximity_197_Factor'] >= 1) & (df['Proximity_328_Factor'] >= 1)
        )
   
        df['Buyout_Factor'] = np.where(
            condition_buyout,
            df[['Proximity_164_Factor', 'Proximity_197_Factor', 'Proximity_328_Factor']].max(axis=1),
            df[['Proximity_164_Factor', 'Proximity_197_Factor', 'Proximity_328_Factor']].min(axis=1)
        )

        df['Market_Value_Avg_Alt'] = np.where(
            df['Occupy_Alt'] == 0,
            np.nan,
       
            np.where(
            df['Alt_Buyout'] > 0,
            df['Alt_Buyout'],
           
            df['Initial_Market_Value'] *
            df['External_Factor'] *
            df['SLR_Factor'] *
            df['Flood_Factor'] *
            df['Buyout_Factor']
            )
        )
   
        df = df.sort_values(by=['SBL', 'Year'])
       
        df['Cost_Alt'] = df.groupby('SBL')['Alt_Buyout'].cumsum() + df.groupby('SBL')['Dam_Alt'].cumsum()
       
        df['Cost_Alt_Private'] = df.groupby('SBL')['Dam_Alt_Private'].cumsum()
       
        df['Cost_Alt_Private_Deduct'] = df.groupby('SBL')['Dam_Alt_Private_Deduct'].cumsum()
   
    final_cols = [col for col in [
        'SBL', 'Year', 'Cost_Alt', 'Cost_Alt_Private', 'Cost_Alt_Private_Deduct',
        'Cost_NFIP', 'Cost_NFIP_Deduct', 'Cost_NFIP_Private', 'Cost_NFIP_Private_Deduct',
        'Occupy_NFIP', 'Occupy_Alt', 'Dam_T_Alt', 'Dam_T_NFIP',
        'Market_Value_Avg_Alt', 'Market_Value_Avg_NFIP'
    ] if col in df.columns]
    return df[final_cols]


In [ ]:

        
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    

def simulate_nfip(seed, Result_245, verbose=False, save_path=None):
    if verbose:
        print(f"  -> Running seed {seed}")

    np.random.seed(seed)
    df = Result_245.copy()

    for Year in range(2025, 2075):
        random_number = np.random.randint(0, 10001)
   
        if random_number < 6303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 0
        elif 6303 <= random_number < 8303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 5
        elif 8303 <= random_number < 9303:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 10
        elif 9303 <= random_number < 9636:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 30
        elif 9636 <= random_number < 9836:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 50
        elif 9836 <= random_number < 9936:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 100
        elif 9936 <= random_number < 9969:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 300
        elif 9969 <= random_number < 9989:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 500
        elif 9989 <= random_number < 9999:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 1000
        else:
            df.loc[df['Year'] == Year, 'Flood_Event'] = 10000
       

    df['Occupy_NFIP'] = np.where(df['SLR'] < df['Year'], 0, 1)

    conditions = [
        (df['SLR'] == df['Year']),
        (df['Flood_Event'] == 0) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 5) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 10) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 30) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 50) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 100) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 300) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 500) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 1000) & (df['SLR'] != df['Year']),
        (df['Flood_Event'] == 10000) & (df['SLR'] != df['Year']),
    ]

    choices_struc = [
        df['ReplaceCost'],
        0,
        df['ReplaceCost'] * df['Dam_Struc_00005'],
        df['ReplaceCost'] * df['Dam_Struc_00010'],
        df['ReplaceCost'] * df['Dam_Struc_00030'],
        df['ReplaceCost'] * df['Dam_Struc_00050'],
        df['ReplaceCost'] * df['Dam_Struc_00100'],
        df['ReplaceCost'] * df['Dam_Struc_00300'],
        df['ReplaceCost'] * df['Dam_Struc_00500'],
        df['ReplaceCost'] * df['Dam_Struc_01000'],
        df['ReplaceCost'] * df['Dam_Struc_10000'],
    ]

    df['Dam_T_Struc'] = np.select(conditions, choices_struc, default=np.nan)

    choices_con = [
        df['ReplaceCost'] * 0.5,
        0,
        df['ReplaceCost'] * df['Dam_Con_00005'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00010'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00030'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00050'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00100'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00300'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_00500'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_01000'] * 0.5,
        df['ReplaceCost'] * df['Dam_Con_10000'] * 0.5,
    ]

    df['Dam_T_Con'] = np.select(conditions, choices_con, default=np.nan)

    df['Dam_T'] = df['Dam_T_Con'] + df['Dam_T_Struc']
    
    df['Dam_T_NFIP'] = np.where(
        df['Occupy_NFIP'] == 1,
        df['Dam_T'],
        0
    )
    
    conditions = [
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'NFIP') & (df['Dam_T_Struc'] >= 277777.78)),
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'NFIP') & (df['Dam_T_Struc'] < 277777.78))
    ]

    choices = [
        250000,
        0.9 * df['Dam_T_Struc']
    ]

    default = 0

    df['Dam_NFIP_Struc'] = np.select(conditions, choices, default=default)
    
    df['Dam_None_Struc'] = np.where(
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'None')),
        df['Dam_T_Struc'],
        0
    )

    conditions = [
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'NFIP') & (df['Dam_T_Con'] >= 111111.11)),
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'NFIP') & (df['Dam_T_Con'] < 111111.11))
    ]

    choices = [
        100000,
        0.9 * df['Dam_T_Con']
    ]

    default = 0

    df['Dam_NFIP_Con'] = np.select(conditions, choices, default=default)
    
    df['Dam_None_Con'] = np.where(
        ((df['Occupy_NFIP'] == 1) & (df['NFIP_Decision_245_FIRM'] == 'None')),
        df['Dam_T_Con'],
        0
    )

    df['Dam_NFIP'] = df['Dam_NFIP_Con'] + df['Dam_NFIP_Struc']
    
    df['Dam_None'] = df['Dam_None_Con'] + df['Dam_None_Struc']
    
    df['Dam_NFIP_Deduct_Struc'] = df['Dam_NFIP_Struc'] * (1/9)

    df['Dam_NFIP_Deduct_Con'] = df['Dam_NFIP_Con'] * (1/9)

    df['Dam_NFIP_Deduct'] = df['Dam_NFIP_Deduct_Con'] + df['Dam_NFIP_Deduct_Struc']

    df['Dam_NFIP_Private_Struc'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        (df['Dam_T_Struc'] - df['Dam_NFIP_Struc'] - df['Dam_NFIP_Deduct_Struc']) * 0.9
    )

    df['Dam_NFIP_Private_Con'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        (df['Dam_T_Con'] - df['Dam_NFIP_Con'] - df['Dam_NFIP_Deduct_Con']) * 0.9
    )

    df['Dam_NFIP_Private'] = df['Dam_NFIP_Private_Con'] + df['Dam_NFIP_Private_Struc']

    df['Dam_NFIP_Private_Deduct_Struc'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        df['Dam_NFIP_Private_Struc'] * (1/9)
    )

    df['Dam_NFIP_Private_Deduct_Con'] = np.where(
        df['Occupy_NFIP'] == 0,
        0,
        df['Dam_NFIP_Private_Con'] * (1/9)
    )

    df['Dam_NFIP_Private_Deduct'] = df['Dam_NFIP_Private_Deduct_Con'] + df['Dam_NFIP_Private_Deduct_Struc']


    df['Flood_Occur_0_Factor'] = np.where(df['Dam_T'] > 0, 0.805, 1)

    df = df.sort_values(by=['SBL', 'Year'])

    df['Dam_T_prev'] = df.groupby('SBL')['Dam_T'].shift(1)

    df['Flood_Occur_1_Factor'] = np.where(df['Dam_T_prev'] > 0, 0.844, 1)

    df = df.drop(columns='Dam_T_prev')


    df['Dam_T_prev2'] = df.groupby('SBL')['Dam_T'].shift(2)

    df['Flood_Occur_2_Factor'] = np.where(df['Dam_T_prev2'] > 0, 0.883, 1)

    df = df.drop(columns='Dam_T_prev2')


    df['Dam_T_prev3'] = df.groupby('SBL')['Dam_T'].shift(3)

    df['Flood_Occur_3_Factor'] = np.where(df['Dam_T_prev3'] > 0, 0.922, 1)

    df = df.drop(columns='Dam_T_prev3')


    df['Dam_T_prev4'] = df.groupby('SBL')['Dam_T'].shift(4)

    df['Flood_Occur_4_Factor'] = np.where(df['Dam_T_prev4'] > 0, 0.961, 1)

    df = df.drop(columns='Dam_T_prev4')

    flood_cols = [
        'Flood_Occur_0_Factor',
        'Flood_Occur_1_Factor',
        'Flood_Occur_2_Factor',
        'Flood_Occur_3_Factor',
        'Flood_Occur_4_Factor'
    ]

    df['Flood_Occur_Factor'] = df[flood_cols].min(axis=1)


    conditions = [
        (df['Flood_Event'] == 0),
        (df['Flood_Event'] == 5) & (df['Prox_00005'] == 'Y'),
        (df['Flood_Event'] == 10) & (df['Prox_00010'] == 'Y'),
        (df['Flood_Event'] == 30) & (df['Prox_00030'] == 'Y'),
        (df['Flood_Event'] == 50) & (df['Prox_00050'] == 'Y'),
        (df['Flood_Event'] == 100) & (df['Prox_00100'] == 'Y'),
        (df['Flood_Event'] == 300) & (df['Prox_00300'] == 'Y'),
        (df['Flood_Event'] == 500) & (df['Prox_00500'] == 'Y'),
        (df['Flood_Event'] == 1000) & (df['Prox_01000'] == 'Y'),
        (df['Flood_Event'] == 10000) & (df['Prox_10000'] == 'Y')
    ]

    choices = [
        1,        
        0.935,  
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935,
        0.935
    ]

    default = 1

    df['Prox_Flood_0_Factor'] = np.select(conditions, choices, default=default)


    df = df.sort_values(by=['SBL', 'Year'])

    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(1)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(1)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(1)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(1)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(1)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(1)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(1)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(1)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(1)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(1)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.948,    
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
        0.948,
    ]

    df['Prox_Flood_1_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(2)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(2)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(2)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(2)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(2)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(2)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(2)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(2)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(2)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(2)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.961,    
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
        0.961,
    ]

    df['Prox_Flood_2_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(3)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(3)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(3)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(3)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(3)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(3)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(3)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(3)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(3)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(3)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.974,    
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
        0.974,
    ]

    df['Prox_Flood_3_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])

    df['Flood_Event_prev'] = df.groupby('SBL')['Flood_Event'].shift(4)
    df['Prox_00005_prev'] = df.groupby('SBL')['Prox_00005'].shift(4)
    df['Prox_00010_prev'] = df.groupby('SBL')['Prox_00010'].shift(4)
    df['Prox_00030_prev'] = df.groupby('SBL')['Prox_00030'].shift(4)
    df['Prox_00050_prev'] = df.groupby('SBL')['Prox_00050'].shift(4)
    df['Prox_00100_prev'] = df.groupby('SBL')['Prox_00100'].shift(4)
    df['Prox_00300_prev'] = df.groupby('SBL')['Prox_00300'].shift(4)
    df['Prox_00500_prev'] = df.groupby('SBL')['Prox_00500'].shift(4)
    df['Prox_01000_prev'] = df.groupby('SBL')['Prox_01000'].shift(4)
    df['Prox_10000_prev'] = df.groupby('SBL')['Prox_10000'].shift(4)

    conditions = [
        (df['Flood_Event_prev'] == 0),
        (df['Flood_Event_prev'] == 5) & (df['Prox_00005_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10) & (df['Prox_00010_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 30) & (df['Prox_00030_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 50) & (df['Prox_00050_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 100) & (df['Prox_00100_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 300) & (df['Prox_00300_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 500) & (df['Prox_00500_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 1000) & (df['Prox_01000_prev'] == 'Y'),
        (df['Flood_Event_prev'] == 10000) & (df['Prox_10000_prev'] == 'Y'),
    ]

    choices = [
        1,        
        0.987,    
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
        0.987,
    ]

    df['Prox_Flood_4_Factor'] = np.select(conditions, choices, default=1)

    df = df.drop(columns=[col for col in df.columns if col.endswith('_prev')])


    flood_cols = [
        'Prox_Flood_0_Factor',
        'Prox_Flood_1_Factor',
        'Prox_Flood_2_Factor',
        'Prox_Flood_3_Factor',
        'Prox_Flood_4_Factor'
    ]

    df['Prox_Flood_Factor'] = df[flood_cols].min(axis=1)


    df['Flood_Factor'] = df[['Flood_Occur_Factor', 'Prox_Flood_Factor']].min(axis=1)

    df['Market_Value_Avg_NFIP'] = np.where(
        df['Occupy_NFIP'] == 0,
        np.nan,
        df['Initial_Market_Value'] *
        df['External_Factor'] *
        df['SLR_Factor'] *
        df['Flood_Factor']
    )


    df = df.sort_values(by=['SBL', 'Year'])
    grouped = df.groupby('SBL')
   
    df['Cost_NFIP'] = grouped['Dam_NFIP'].cumsum()


    df['Cost_NFIP_Deduct'] = grouped['Dam_NFIP_Deduct'].cumsum()


    df['Cost_NFIP_Private'] = grouped['Dam_NFIP_Private'].cumsum()


    df['Cost_NFIP_Private_Deduct'] = grouped['Dam_NFIP_Private_Deduct'].cumsum()

    
    df['Cost_None'] = grouped['Dam_None'].cumsum()
   

    final_cols = [col for col in [
        'SBL', 'Year',
        'Cost_NFIP', 'Cost_NFIP_Deduct', 'Cost_NFIP_Private', 'Cost_NFIP_Private_Deduct',
        'Cost_None',
        'Occupy_NFIP', 'Dam_T_NFIP',
        'Market_Value_Avg_NFIP'
    ] if col in df.columns]
    return df[final_cols]

In [7]:
Jamaica_Bay_Parcels = pd.read_csv(r"C:\Users\mrenna\OneDrive - University of Maryland\Desktop\Project Files\Jamaica_Bay_Parcels_Set_Up_COMPLETE_Correction_DECADE.csv")

In [8]:
print(Jamaica_Bay_Parcels.columns)

Index(['SBL', 'YR_BLT', 'BLDG_STYLE', 'Neighborhood', 'FIRM_FLD_ZONE',
       'PFIRM_FLD_ZONE', 'BLD_STORY', 'Land_MKT_VAL', 'Total_MKT_VAL',
       'ReplaceCost', 'z_grade', 'z_floor', 'subgrade', 'BFE', 'DDF_Assign',
       'DDF_Dam', 'Census_Tract', 'Mu', 'Sigma', 'Income_House',
       'Income_Total', 'SLR_245', 'SLR_585', 'Year', 'Mean_Forecast',
       'Std_Dev_Forecast', 'Prox_00005_245', 'Prox_00010_245',
       'Prox_00030_245', 'Prox_00050_245', 'Prox_00100_245', 'Prox_00300_245',
       'Prox_00500_245', 'Prox_01000_245', 'Prox_10000_245', 'Prox_00005_585',
       'Prox_00010_585', 'Prox_00030_585', 'Prox_00050_585', 'Prox_00100_585',
       'Prox_00300_585', 'Prox_00500_585', 'Prox_01000_585', 'Prox_10000_585',
       'Zone_Decade_245', 'Zone_Decade_585', 'SLR_245_Factor',
       'SLR_585_Factor', 'Flood_245_Factor', 'Flood_585_Factor',
       'Dam_Con_00005_245', 'Dam_Con_00010_245', 'Dam_Con_00030_245',
       'Dam_Con_00050_245', 'Dam_Con_00100_245', 'Dam_Con_00300_245',